# Llama answer bottleneck — Llama-3.1 8B → Llama-3.2 1B, graft (11, 17)

**Question.** On Gemma the task map confers 0.886 and the answer bottleneck (EXP26) recovers 96.9% of the matched task map. The objection: Gemma's map is saturated (six times the data adds only 0.032, App. A.18; rank 16 already gives 0.875 of 0.887), so a bottleneck matching it may only show there was nothing left to gain. Llama's map is not saturated: six times the data takes it from 0.217 to 0.586 (2.7×, App. A.18), and a per-problem free vector with gold access drives the recipient to 0.89 (§5). (`consolidated_results` also lists a 0.990 oracle, but that map was fit on the eval items themselves, so it bounds the writer side, not what a held-out map can extract.) If a channel carrying **only the donor's answer** accounts for Llama's conferral where more was available, "answers, not computation" holds beyond the saturated case.

**What counts against the thesis.** The paper (§1) defines computation transfer to include "a recipient that uses the stitched layer to compute answers or digits beyond what the donor had in its graft site". On Llama the scored first token is the whole number, so the recipient can fill in a digit inside it (on Gemma the first token is one digit). Results showing the recipient computing digits from its own prompt (masked codes, Part D) therefore count **against** the thesis on Llama, not as a neutral alternative.

**Part A — reproduce the paper's Llama eval.** Cells A1–A3 are copied verbatim from `Llama_eval_EXP13-19` (same models, layers, problem ranges, seeds, recipe). A4 checks the numbers against `consolidated_results (10).json` (bin n = 626, task first 0.278, recon 0.019, shuffle 0.005, task full 0.291) and stops the run if they don't reproduce. Note: 0.278 is the **seed-0** map (the paper's text calls the task-map first-token figure "five-seed pooled"; the code scores seed 0). A4 also prints the across-seed mean.

**Part B — bottleneck on the same 3,000 training problems and the same 626-item bin.** The EXP26 recipe (reader and writer on disjoint halves, matched task map on the writer half). Readers, each feeding the writer nothing but answer scores:
- `token` — one softmax over the first-token classes in the training set (~700 at 3,000 problems, ~825 at 18,000; 841 are possible). The literal port of EXP26: class-starved at 1,500 reader items, and ~700 wide rather than Gemma's 9.
- `digits` — three 10-way softmaxes (hundreds, tens, ones): 30 scores, 27 free dimensions (Gemma's K = 9 has 8). The same factorisation as the paper's Llama whole-answer probe (`Emergence_Llama`, CELL S1). Gold = three concatenated one-hots. The gold writer is additive in the digits, so its ceiling is not 1.0 by construction.
- `digits_mlp` — the same channel, each digit read by the paper's MLP probe (`probe_mlp`, `Followup3_MLPprobe_Llama`).
- `…:donor` — the reader is trained on the **donor's own top-1 answer** instead of the gold one. The donor is right on only ~32% of training problems, so a gold-label reader could learn to compute answers the donor never had; the `:donor` readers read what the donor's state encodes. **Read these first.**
- `…:self` — leakage control: the reader reads the **recipient's own** layer-11 states. If this confers, a reader + writer can manufacture conferral from what the recipient already has.

Plus: the reader's argmax only (one-hot through the gold writer), masked digit codes (gold code with the tens or ones blanked — does the recipient fill the digit in?), a rank-27 matched map (same width as the digit channel), the full-answer task-map reference, paired differences with bootstrap intervals, and splits by writer-half class coverage and by operand-swap twins in training (b × a + c, same answer).

**Part C — the same test at 18,000 training problems.** This notebook's set nests Part A's 3,000 and excludes all 2,000 generated eval expressions. The paper's own 18k run (EXP28: 0.217 → 0.586) used a different draw, two epochs and excluded only the 642 donor-solved eval items, so expect different numbers. Swap twins of bin problems are much more common at 18k, so part of any 3k → 18k gain can be near-lookup; read the twin split.

**Part D — prompt-mismatch test** (Llama port of Experiment A in `Subspace-ablation-followups/48GB_AnswerSubspace_followups`, the paper's §6.2 test). The recipient reads P′ = (a, b, c + δ) while the graft comes from the donor's state on P = (a, b, c). Two δ rules: `leading` (Experiment A's rule, leading digit flips) and `far` (tens moves by ≥ 2, ones changes, y′ not a multiple of ten), with a proximity null and a 20-draw chance null. Only completion that depends on c is detectable, since a and b are shared.

Everything new is single-seed (bottlenecks, matched maps, the 18k map); the 3k task map has five seeds.

**Run.** Cell 1, restart the kernel, then everything from Cell 2 down. Put your token in the `HF_TOKEN` environment variable or paste it into the login cell (Llama is gated). One 48 GB card (8B donor in bf16 + 1B recipient). Rough time 2–3 hours (Part A 20–30 min, Part B 20–30 min, Part C 60–100 min, Part D ~10 min) plus ~19 GB of downloads; checkpoints are written after A4, B, C and D. Trim `BN_READERS` to shorten it. `SMOKE_TEST = True` runs the plumbing in a few minutes; its numbers mean nothing.

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) — same pins as Llama_eval_EXP13-19 ===
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
# >>> RESTART KERNEL after this, then run every cell below in order. <<<

# --- OPTIONAL: Blackwell / sm_120 torch fix (uncomment ONLY if the next cell reports (12, 0)) ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128
# !pip uninstall -y torchvision torchaudio


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
# verify the resolved stack before spending GPU time
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")   # must be set before transformers / huggingface_hub load
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "(48 GB recommended)")

torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)
VRAM GB: 47.7 (48 GB recommended)


In [2]:
# === CELL 2: imports, set_submodule shim, global config — Llama_eval_EXP13-19 values ===
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")   # also set in the verify cell, which imports transformers first
import json, math, random, re, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "meta-llama/Llama-3.2-1B"
MODEL_9B = "meta-llama/Llama-3.1-8B"

SMOKE_TEST = False            # True = ~5 min plumbing check only; its numbers are meaningless

SINGLE_PAIR = (11, 17)        # Llama pair (2B graft, 9B donor): the CKA table after EXP1 (L11<->L17 CKA 0.838)
LAYER_PAIRS = [(11, 16), (12, 18), (13, 20), (14, 22)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

# Llama problem range. In Llama_eval_EXP13-19 these five lines sit inside the full-run branch, so a
# smoke test there silently fell back to the Gemma ranges; here they apply to both modes.
ARITH_A_HI, ARITH_B_HI, ARITH_C_HI = 30, 30, 40
ARITH_ANS_MIN, ARITH_ANS_MAX = 100, 999      # every answer is 3 digits = one Llama token

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
    N_ARITH_TRAIN_SCALED = 1200
    BN_READER_STEPS, BN_WRITER_EPOCHS = 100, 2
    BN_MLP_STEPS = 200
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6
    BOOT_B = 10000
    N_ARITH_TRAIN_SCALED = 18000              # = the data-scale appendix
    BN_READER_STEPS, BN_WRITER_EPOCHS = 600, 8   # = EXP26
    BN_MLP_STEPS = 800                           # = probe_mlp in Followup3_MLPprobe_Llama

ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8

# ---- bottleneck knobs (EXP26 defaults) ----
RUN_BASE_BOTTLENECK   = True      # Part B
RUN_SCALED_BOTTLENECK = True      # Part C
RUN_PERTURB           = True      # Part D
STOP_ON_REPRO_FAIL    = True      # stop after A4 if Part A does not reproduce (full runs only)
# readers: kind[:variant]. ':donor' = labels are the donor's own top-1 answer (what its state
# encodes) instead of gold; ':self' = the reader reads the RECIPIENT's own L11 states (leakage
# control: how much conferral a reader+writer can manufacture from what the recipient already has)
BN_READERS     = ("token", "digits", "digits_mlp", "digits:donor", "digits_mlp:donor",
                  "digits:self", "digits_mlp:self")
BN_MATCHED_RANK = 27              # rank-matched comparator = the digit channel's free dimensions
BN_READER_FRAC = 0.5              # reader half / writer half, disjoint
BN_READER_LR   = 1e-2
BN_WRITER_LR   = 1e-3
BN_SOFTMAX     = True             # scores are softmax (EXP26); False = raw logits, a wider channel
BN_SEED        = 0
# digits_mlp reader = probe_mlp from Followup3_MLPprobe_Llama, one net per digit
BN_MLP_HIDDEN, BN_MLP_LR, BN_MLP_WD = 256, 1e-3, 1e-3
# Part D: delta for P' = (a, b, c + delta) is searched over 1 <= |delta| < ARITH_C_HI
P_RULES = ("leading", "far")      # 'leading' = Experiment A's rule; 'far' = per-digit scoring
P_SHUFFLE_SEED = 1234             # = A_SHUFFLE_SEED in Experiment A

RESULTS = {}
PER_ITEM = {}                     # per-item booleans on the bin, saved for later paired tests
BN_MODELS = {}                    # trained (reader, pred-writer) per arm and reader, for Part D

def save_partial(tag):
    """Checkpoint everything so far, so a crash late in the run does not lose earlier parts."""
    fn = "llama_bottleneck_results" + ("_SMOKE" if SMOKE_TEST else "") + "_partial.json"
    try:
        with open(fn, "w") as fh:
            json.dump({"_last_completed": tag, **RESULTS, "_per_item_so_far": PER_ITEM}, fh, default=str)
        print(f"[checkpoint after {tag} -> {fn}]")
    except Exception as e:
        print("checkpoint failed:", e)
print("SMOKE_TEST =", SMOKE_TEST)

SMOKE_TEST = False


In [ ]:
# === CELL 3: Hugging Face login (Llama is gated) ===
# Prefer the HF_TOKEN environment variable so no token is ever saved into this notebook.
from huggingface_hub import login
login(os.environ.get("HF_TOKEN") or "")

In [5]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

from fractions import Fraction
def mcnemar(a, b):
    """Exact two-sided McNemar (as in EXP26). Returns (n01, n10, p):
    n01 = a wrong & b right, n10 = a right & b wrong."""
    n01 = sum(1 for x, y in zip(a, b) if (not x) and y)
    n10 = sum(1 for x, y in zip(a, b) if x and (not y))
    n = n01 + n10
    if n == 0: return n01, n10, 1.0
    k = min(n01, n10)
    p = Fraction(2 * sum(math.comb(n, i) for i in range(k + 1)), 2 ** n)
    return n01, n10, min(1.0, float(p))

In [6]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook
 
def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m
 
def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2
 
# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    a_hi = globals().get("ARITH_A_HI", 40); b_hi = globals().get("ARITH_B_HI", 40)
    c_hi = globals().get("ARITH_C_HI", 99)
    ans_max = globals().get("ARITH_ANS_MAX", None)
    ans_min = globals().get("ARITH_ANS_MIN", None)
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*200:
        tries += 1
        a, b, c = rng.randint(2, a_hi), rng.randint(2, b_hi), rng.randint(1, c_hi)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if ans_max is not None and ans > ans_max: continue
        if ans_min is not None and ans < ans_min: continue
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out
 
# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    # take the FIRST #### marker and the number right after it; ignore any rambling after
    m = _re.search(r"####\s*\$?(-?[\d,]+(?:\.\d+)?)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: return None
    # no #### marker -> the model didn't produce a clean final answer; count as wrong, don't
    # guess from trailing numbers (that's what caused inconsistent parsing across conditions)
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None
 
# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top
 
# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)
 
@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok
 
print("helpers defined")

helpers defined


In [7]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
 
# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
 
model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x2B layers,",
      model_9b.config.num_hidden_layers, "x9B layers |",
      "9B quantized" if QUANTIZE_9B else "9B bf16")
 
# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "9B produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl
 
# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

... (8 outputs left out)

loaded: 16 x2B layers, 32 x9B layers | 9B bf16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

## Part A — reproduce the Llama eval
A1–A3 are `Llama_eval_EXP13-19` cells 8–10 unchanged, apart from one added line in A1 that records every generated eval expression before the donor filter (Part C excludes them all from its training set).

In [8]:
# [A1] # === CELL 8: EXP2 setup — arithmetic states, native correctness, reconstruction map ===
train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")

X9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in train]); X9t = X9t[L9_SINGLE]
X9e, nine_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evalp]); X9e = X9e[L9_SINGLE]
EVAL_EXPRS_ALL = {p["expr"] for p in evalp}   # [added] all 2000 generated eval exprs, pre-filter
keep = [i for i in range(len(evalp)) if nine_top[i] == evalp[i]["tok"]]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  kept {len(evalp)} the 9B solves")

X2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in train]); X2t = X2t[L2_SINGLE]
X2e, two_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evalp]); X2e = X2e[L2_SINGLE]
solvable = [two_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
print(f"  2B natively solves {len(solv)}/{len(evalp)} (unsolvable bin n={len(unsolv)})")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
recon_map = (mu9.to(DEVICE), mu2.to(DEVICE), Wr.to(DEVICE))
def map_recon(x9): m9, m2, W = recon_map; return (x9.to(DEVICE)-m9)@W + m2

arith: 3000 train, 2000 eval
  kept 642 the 9B solves
  2B natively solves 16/642 (unsolvable bin n=626)


In [9]:
# [A1b] sanity: on Llama the first answer token IS the whole 3-digit answer
_bad_tr = sum(tokenizer.decode([p["tok"]]).strip() != str(p["ans"]) for p in train)
_bad_ev = sum(tokenizer.decode([p["tok"]]).strip() != str(p["ans"]) for p in evalp)
print(f"first token != whole answer: train {_bad_tr}/{len(train)}, eval {_bad_ev}/{len(evalp)}  (want 0, 0)")
print(f"answer classes in train: {len({p['tok'] for p in train})}  (the range admits 841)")

first token != whole answer: train 0/3000, eval 0/642  (want 0, 0)
answer classes in train: 695  (the range admits 841)


In [10]:
# [A2] # === CELL 9: EXP3 — train the task-supervised map, 5 seeds (the only stochastic part) ===
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
task_maps = []
model_2b.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9 = X9t[sub].to(DEVICE)
                _graft["vec"] = (x9 - mu9d) @ W + b
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}")
model_2b.requires_grad_(True)

  seed 0: final batch CE 0.964
  seed 1: final batch CE 0.579
  seed 2: final batch CE 0.644
  seed 3: final batch CE 0.733
  seed 4: final batch CE 0.512


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): Lla

In [11]:
# [A3] # === CELL 10: EXP2+3 eval — reconstruct vs task, first-token AND full-answer, with CIs ===
@torch.inference_mode()
def first_token_confer(map_fn, idxs):
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            _graft["vec"] = map_fn(X9e[sub])
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
def full_confer(map_fn, idxs):
    vecs = list(map_fn(X9e[idxs]).cpu())
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=vecs)
 
def map_task(i):
    W, b = task_maps[i]
    return lambda x9: (x9.to(DEVICE)-mu9d)@W + b
shuf = torch.randperm(len(evalp))
@torch.inference_mode()
def shuffle_confer(idxs):  # task map fed the WRONG problem's 9B state (specificity control)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
    W, b = task_maps[0]
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            donor = X9e[shuf[i:i+len(sub)]].to(DEVICE)
            _graft["vec"] = (donor - mu9d)@W + b
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
 
# recon_cos (continuous metric -> bootstrap CI); task map should sit much lower
rc_recon = F.cosine_similarity(map_recon(X9e).cpu(), X2e, dim=1).numpy()
rc_task = F.cosine_similarity(((X9e.to(DEVICE)-mu9d)@task_maps[0][0]+task_maps[0][1]).cpu(), X2e, dim=1).numpy()
arr = {"recon_cos_recon": fmt(bootstrap_ci(rc_recon)),
       "recon_cos_task": fmt(bootstrap_ci(rc_task))}
 
# headline bin: UNSOLVABLE — full-answer + the 5-seed treatment live here
if unsolv:
    arr["native_unsolv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in unsolv], vecs=None)))
    arr["recon_unsolv_full"] = fmt(wilson_bools(full_confer(map_recon, unsolv)))
    arr["recon_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_recon, unsolv)))
    seed_full = [float(np.mean(full_confer(map_task(s), unsolv))) for s in range(len(task_maps))]
    arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
    arr["task_unsolv_first_pooled"] = fmt(wilson_bools(first_token_confer(map_task(0), unsolv)))
    arr["shuffle_unsolv_first"] = fmt(wilson_bools(shuffle_confer(unsolv)))
# sanity bin: SOLVABLE — first-token + full-answer (recon and task seed 0)
if solv:
    arr["native_solv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in solv], vecs=None)))
    arr["recon_solv_first"] = fmt(wilson_bools(first_token_confer(map_recon, solv)))
    arr["recon_solv_full"] = fmt(wilson_bools(full_confer(map_recon, solv)))
    arr["task_solv_first"] = fmt(wilson_bools(first_token_confer(map_task(0), solv)))
    arr["task_solv_full"] = fmt(wilson_bools(full_confer(map_task(0), solv)))
RESULTS["arithmetic"] = arr
print("EXP2/3:", json.dumps(arr, indent=2))

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


EXP2/3: {
  "recon_cos_recon": "0.977 [0.977, 0.978]",
  "recon_cos_task": "0.261 [0.259, 0.264]",
  "native_unsolv_full": "0.000 [0.000, 0.006]",
  "recon_unsolv_full": "0.019 [0.011, 0.033]",
  "recon_unsolv_first": "0.019 [0.011, 0.033]",
  "task_unsolv_full_acrossseed": "0.291 [0.276, 0.305]",
  "task_unsolv_first_pooled": "0.278 [0.244, 0.314]",
  "shuffle_unsolv_first": "0.005 [0.002, 0.014]",
  "native_solv_full": "1.000 [0.806, 1.000]",
  "recon_solv_first": "0.250 [0.102, 0.495]",
  "recon_solv_full": "0.250 [0.102, 0.495]",
  "task_solv_first": "0.188 [0.066, 0.430]",
  "task_solv_full": "0.188 [0.066, 0.430]"
}


In [12]:
# === [A4] reproduction check against consolidated_results (10).json ===
# Counts should match exactly (states and argmax are deterministic up to bf16 kernels). The task map
# is trained on the GPU, so allow a few items of drift there.
def _pt(s): return float(str(s).split()[0])
PAPER = {"n_eval_kept": 642, "n_unsolv": 626, "n_solv": 16,
         "recon_cos_recon": 0.977, "native_unsolv_full": 0.000, "recon_unsolv_first": 0.019,
         "shuffle_unsolv_first": 0.005, "task_unsolv_first_pooled": 0.278,
         "task_unsolv_full_acrossseed": 0.291}
TOL = {"recon_cos_recon": 0.003, "native_unsolv_full": 0.003, "recon_unsolv_first": 0.006,
       "shuffle_unsolv_first": 0.010, "task_unsolv_first_pooled": 0.030,
       "task_unsolv_full_acrossseed": 0.030}
got = {"n_eval_kept": len(evalp), "n_unsolv": len(unsolv), "n_solv": len(solv)}
got.update({k: _pt(RESULTS["arithmetic"][k]) for k in TOL})
checks = {}
for k, v in PAPER.items():
    if k.startswith("n_"):
        ok = got[k] == v
        tag = "PASS" if ok else ("WARN" if abs(got[k] - v) <= 2 else "FAIL")   # a bf16 argmax tie can flip 1-2 items
    else:
        ok = abs(got[k] - v) <= TOL[k]; tag = "PASS" if ok else "FAIL"
    checks[k] = {"paper": v, "this_run": got[k], "status": tag}
    print(f"  {tag}  {k:30s} paper {v:<7} this run {got[k]}")
print("(run cells in order: re-running A3 after Part B changes the shuffle control, because the MLP "
      "reader's initialisation draws from torch's CPU RNG)")
if SMOKE_TEST:
    print("(SMOKE_TEST: sizes differ from the paper, so FAILs here are expected)")
RESULTS["reproduction_check"] = checks
save_partial("A4")
if STOP_ON_REPRO_FAIL and not SMOKE_TEST and any(c["status"] == "FAIL" for c in checks.values()):
    raise RuntimeError("Part A did not reproduce the paper's Llama numbers (see FAIL lines above). "
                       "Stopping before Parts B-D. Set STOP_ON_REPRO_FAIL = False to continue anyway.")

# extra (not in the paper table): first-token conferral for every seed, not just seed 0
_seed_first = [float(np.mean(first_token_confer(map_task(s), unsolv))) for s in range(len(task_maps))]
RESULTS["arithmetic"]["task_unsolv_first_acrossseed"] = fmt(across_seed_ci(_seed_first))
RESULTS["arithmetic"]["task_unsolv_first_per_seed"] = [round(x, 3) for x in _seed_first]
print("task first-token per seed:", RESULTS["arithmetic"]["task_unsolv_first_per_seed"])

  PASS  n_eval_kept                    paper 642     this run 642
  PASS  n_unsolv                       paper 626     this run 626
  PASS  n_solv                         paper 16      this run 16
  PASS  recon_cos_recon                paper 0.977   this run 0.977
  PASS  native_unsolv_full             paper 0.0     this run 0.0
  PASS  recon_unsolv_first             paper 0.019   this run 0.019
  PASS  shuffle_unsolv_first           paper 0.005   this run 0.005
  PASS  task_unsolv_first_pooled       paper 0.278   this run 0.278
  PASS  task_unsolv_full_acrossseed    paper 0.291   this run 0.291
(run cells in order: re-running A3 after Part B changes the shuffle control, because the MLP reader's initialisation draws from torch's CPU RNG)
[checkpoint after A4 -> llama_bottleneck_results_partial.json]
task first-token per seed: [0.278, 0.292, 0.304, 0.3, 0.28]


## The bottleneck machinery
EXP26 from `Stitch_STANDALONE_EXP26_27`, parameterised by training set and reader. For the `token` reader the recipe is EXP26's except that the reader trains on the GPU (same objective, 600 steps, lr 1e-2). Additions: the digit readers and their `:donor` / `:self` variants, the reader-argmax condition, the masked-code writers, the rank-27 matched map, the full-answer task-map reference (EXP26's `_ref_task_full`), paired-difference intervals, and the coverage and twin splits. Gold, uniform and masked writers depend only on the code, not the reader, so they are trained once per arm and shared. The matched maps train before the writers here (after them in EXP26); every trainer reseeds, so the order has no effect.

In [13]:
# === CELL B0: answer-bottleneck machinery ===
import collections
#   h_D --[frozen linear reader, reader half]--> scores z --[linear writer, writer half]--> graft
# The writer sees ONLY z. Every condition is scored on the same unsolvable bin as Part A.

def _partner_map(idxs, seed):
    """Derangement of the bin with distinct answer tokens (verbatim from EXP26)."""
    rng = random.Random(seed)
    pool = list(idxs)
    for _ in range(200):
        perm = pool[:]; rng.shuffle(perm)
        bad = [k for k in range(len(pool))
               if evalp[perm[k]]["tok"] == evalp[pool[k]]["tok"]]
        if not bad: return dict(zip(pool, perm)), 0
        for k in bad:                                    # greedy repair
            for m2 in range(len(pool)):
                if (evalp[perm[m2]]["tok"] != evalp[pool[k]]["tok"] and
                    evalp[perm[k]]["tok"]  != evalp[pool[m2]]["tok"]):
                    perm[k], perm[m2] = perm[m2], perm[k]; break
        bad = [k for k in range(len(pool))
               if evalp[perm[k]]["tok"] == evalp[pool[k]]["tok"]]
        if not bad: return dict(zip(pool, perm)), 0
    return dict(zip(pool, perm)), len(bad)
PARTNER, N_COLLIDE = _partner_map(unsolv, BN_SEED + 7)
print(f"shuffled pairing: {len(PARTNER)} pairs, {N_COLLIDE} unavoidable same-token collisions")

def _digits(ans):
    """Place-value digits (hundreds, tens, ones) of a 3-digit answer."""
    s = str(int(ans))
    assert len(s) == 3, f"expected a 3-digit answer, got {ans}"
    return [int(c) for c in s]

@torch.inference_mode()
def _confer_vecfn(vec_fn, idxs):
    """vec_fn(sub) -> [len(sub), d2] on DEVICE. First-token conferral, grafted per batch."""
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i + ARITH_BATCH]
            v = vec_fn(sub)
            assert v.shape[0] == len(sub), f"graft batch {v.shape[0]} != {len(sub)}"
            _graft["vec"] = v
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        h.remove(); _graft["vec"] = None
    return out

@torch.inference_mode()
def _emits_which(vec_fn, idxs, partner):
    """Top token vs the PARTNER item's answer (did the channel deliver the other answer?)."""
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i + ARITH_BATCH]
            _graft["vec"] = vec_fn(sub)
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[partner[sub[k]]]["tok"] for k in range(len(sub))]
    finally:
        h.remove(); _graft["vec"] = None
    return out

def _full_vecfn(vec_fn, idxs):
    V = torch.cat([vec_fn(idxs[i:i + ARITH_BATCH]).detach().cpu()
                   for i in range(0, len(idxs), ARITH_BATCH)], 0)
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=list(V))

def _digit_code(answers, mask=None):
    """Three concatenated one-hots (hundreds, tens, ones). mask = index of a head to blank to
    uniform 0.1, used by the masked-code writers that test whether the recipient fills it in."""
    d = torch.tensor([_digits(a) for a in answers], device=DEVICE)
    heads = [F.one_hot(d[:, k], 10).float() for k in range(3)]
    if mask is not None: heads[mask] = torch.full_like(heads[mask], 0.1)
    return torch.cat(heads, -1)

def _answer_of_token(t):
    """Decoded 3-digit answer of a token id, or None (used for the donor's own top-1 answer)."""
    s = tokenizer.decode([t]).strip()
    return int(s) if (len(s) == 3 and s.isdigit()) else None

class Reader:
    """Frozen answer reader. Trained on rows X (donor states, or the recipient's own states for the
    ':self' control) with labels y (first-token ids for 'token', integer answers for the digit kinds).
      'token'      : linear, K-way over the first-token classes of the training set (EXP26 port)
      'digits'     : linear, three 10-way heads (hundreds, tens, ones), summed CE (= three probes)
      'digits_mlp' : three 10-way heads, each the paper's probe_mlp (Followup3_MLPprobe_Llama):
                     standardise -> Linear(d, 256) -> GELU -> Linear(256, 10), AdamW lr 1e-3,
                     weight decay 1e-3, 800 full-batch steps, torch.manual_seed(0) per head."""
    def __init__(self, kind, X, y, tokset=None):
        self.kind = kind
        if kind == "token":
            self.TOKSET = tokset
            self.tok2cls = {t: i for i, t in enumerate(self.TOKSET)}
            self.width = len(self.TOKSET)
            Y = torch.tensor([self.tok2cls[t] for t in y])
        elif kind in ("digits", "digits_mlp"):
            self.width = 30
            Y = torch.tensor([_digits(a) for a in y])
        else:
            raise ValueError(kind)
        X = X.float().to(DEVICE); Y = Y.to(DEVICE)
        if kind == "digits_mlp":
            self.mu, self.sd = X.mean(0), X.std(0).clamp_min(1e-6)
            A = (X - self.mu) / self.sd
            self.nets = []
            for k in range(3):
                torch.manual_seed(BN_SEED)
                net = nn.Sequential(nn.Linear(X.shape[1], BN_MLP_HIDDEN), nn.GELU(),
                                    nn.Linear(BN_MLP_HIDDEN, 10)).to(DEVICE)
                opt = torch.optim.AdamW(net.parameters(), lr=BN_MLP_LR, weight_decay=BN_MLP_WD)
                for _ in range(BN_MLP_STEPS):
                    opt.zero_grad(); F.cross_entropy(net(A), Y[:, k]).backward(); opt.step()
                net.eval().requires_grad_(False); self.nets.append(net)
        else:
            mu = X.mean(0); A = X - mu
            W = torch.zeros(X.shape[1], self.width, device=DEVICE, requires_grad=True)
            b = torch.zeros(self.width, device=DEVICE, requires_grad=True)
            opt = torch.optim.Adam([W, b], lr=BN_READER_LR)
            for _ in range(BN_READER_STEPS):
                opt.zero_grad(); self._loss(A @ W + b, Y).backward(); opt.step()
            self.mu, self.W, self.b = mu.detach(), W.detach(), b.detach()
        del X, A

    def _loss(self, logits, Y):
        if self.kind == "token": return F.cross_entropy(logits, Y)
        return sum(F.cross_entropy(logits[:, 10*k:10*k + 10], Y[:, k]) for k in range(3))

    @torch.no_grad()
    def logits(self, x):                      # frozen: no grad path into the reader
        if self.kind == "digits_mlp":
            A = (x.float().to(DEVICE) - self.mu) / self.sd
            return torch.cat([net(A) for net in self.nets], -1)
        return (x.float().to(DEVICE) - self.mu) @ self.W + self.b

    def scores(self, x):
        z = self.logits(x)
        if not BN_SOFTMAX: return z
        if self.kind == "token": return F.softmax(z, -1)
        return torch.cat([F.softmax(z[:, 10*k:10*k + 10], -1) for k in range(3)], -1)

    def hard(self, x):
        """One-hot of the reader's argmax (three one-hots for digits): the DECODED answer only."""
        z = self.logits(x)
        if self.kind == "token": return F.one_hot(z.argmax(-1), self.width).float()
        return torch.cat([F.one_hot(z[:, 10*k:10*k + 10].argmax(-1), 10).float() for k in range(3)], -1)

    def gold(self, items):
        if self.kind == "token":
            # answers with no train class get a one-hot at class 0 (wrong), exactly as EXP26
            y = torch.tensor([self.tok2cls.get(p["tok"], 0) for p in items], device=DEVICE)
            return F.one_hot(y, self.width).float()
        return _digit_code([p["ans"] for p in items])

    def uniform(self, n):                     # bias-only / no-information control
        v = 1.0 / self.width if self.kind == "token" else 0.1
        return torch.full((n, self.width), v, device=DEVICE)

    def correct_on(self, xe, items):
        """Reader argmax vs gold on eval rows. Digit kinds: joint (all three right) plus per digit."""
        z = self.logits(xe)
        if self.kind == "token":
            pred = z.argmax(-1).cpu().tolist()
            return [pred[k] == self.tok2cls.get(p["tok"], -1) for k, p in enumerate(items)], None
        per = []
        for k in range(3):
            pk = z[:, 10*k:10*k + 10].argmax(-1).cpu().tolist()
            per.append([pk[n] == _digits(p["ans"])[k] for n, p in enumerate(items)])
        joint = [all(per[k][n] for k in range(3)) for n in range(len(items))]
        return joint, per

def _train_writer(z_fn, width, train_items, X2tr, w_idx, tag):
    """EXP26 writer: graft = z @ B + b, trained with CE on the gold first token through the
    frozen recipient. B ~ N(0, .02), b starts at the recipient's mean state."""
    torch.manual_seed(BN_SEED)
    Bw = torch.empty(width, X2tr.shape[1], device=DEVICE)
    with torch.no_grad(): Bw.normal_(0.0, 0.02)
    Bw.requires_grad_(True)
    bw = X2tr.mean(0).to(DEVICE).clone().float().requires_grad_(True)
    opt = torch.optim.Adam([Bw, bw], lr=BN_WRITER_LR)
    model_2b.requires_grad_(False)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    try:
        for ep in range(BN_WRITER_EPOCHS):
            idx = list(w_idx); random.Random(1000 + ep).shuffle(idx)
            tot, seen = 0.0, 0
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s + ARITH_BATCH]
                _graft["vec"] = z_fn(sub) @ Bw + bw
                ids, m = left_pad([train_items[k]["ids"] for k in sub], tokenizer.pad_token_id)
                logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train_items[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(logits, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
                tot += float(loss.item()) * len(sub); seen += len(sub)
            if ep == BN_WRITER_EPOCHS - 1 or ep % 4 == 0:
                print(f"     writer[{tag}] epoch {ep}  CE = {tot/max(1, seen):.4f}")
    finally:
        handle.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
    return Bw.detach(), bw.detach()

def _train_matched_task(train_items, X9tr, X2tr, w_idx, rank=None):
    """EXP26 matched-budget task map: ridge on the writer half, then CE for the writer's epochs.
    rank=r trains a rank-r map (x - mu) @ A @ B + b instead, initialised from the rank-r SVD of
    the same ridge solution and then TRAINED (not truncated after training): the comparator with
    the same width as a 30-score digit channel."""
    torch.manual_seed(BN_SEED)
    _mu9, _mu2, _Wr0 = fit_ridge(X9tr[w_idx], X2tr[w_idx])
    _mu9d = _mu9.to(DEVICE)
    bm = _mu2.clone().to(DEVICE).float().requires_grad_(True)
    if rank is None:
        params = [_Wr0.clone().to(DEVICE).requires_grad_(True)]
        Wfn = lambda: params[0]
    else:
        U, Sv, Vh = torch.linalg.svd(_Wr0, full_matrices=False)
        r = min(rank, Sv.numel()); sq = Sv[:r].sqrt()
        params = [(U[:, :r] * sq).to(DEVICE).requires_grad_(True),
                  (sq[:, None] * Vh[:r]).to(DEVICE).requires_grad_(True)]
        Wfn = lambda: params[0] @ params[1]
    opt = torch.optim.Adam(params + [bm], lr=BN_WRITER_LR)
    seed_base = 2000 if rank is None else 3000
    model_2b.requires_grad_(False)
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    try:
        for ep in range(BN_WRITER_EPOCHS):
            idx = list(w_idx); random.Random(seed_base + ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s + ARITH_BATCH]
                _graft["vec"] = (X9tr[sub].to(DEVICE) - _mu9d) @ Wfn() + bm
                ids, m = left_pad([train_items[k]["ids"] for k in sub], tokenizer.pad_token_id)
                logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train_items[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(logits, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
    finally:
        h.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
    with torch.no_grad(): Wm = Wfn().detach()
    bm = bm.detach()
    return lambda sub: (X9e[sub].to(DEVICE) - _mu9d) @ Wm + bm

def _twin(expr):
    """'a * b + c' -> 'b * a + c' (same answer, different string), or None when a == b."""
    a, b, c = (int(x) for x in re.match(r"^(\d+) \* (\d+) \+ (\d+)$", expr).groups())
    return None if a == b else f"{b} * {a} + {c}"

def _rate_on(v, ks):
    return round(sum(v[k] for k in ks) / len(ks), 3) if ks else None

def paired_diff_ci(a, b, seed=0):
    """Mean of (b - a) over paired items with a percentile bootstrap 95% interval."""
    d = np.asarray(b, float) - np.asarray(a, float)
    if len(d) == 0: return (float("nan"),) * 3
    rng = np.random.default_rng(seed)
    m = d[rng.integers(0, len(d), (BOOT_B, len(d)))].mean(1)
    lo, hi = np.percentile(m, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)

def _paired(a_task, b_bn):
    n01, n10, p = mcnemar(a_task, b_bn)
    return {"task_only": n10, "bottleneck_only": n01, "mcnemar_p": p,
            "diff_bottleneck_minus_task": fmt(paired_diff_ci(a_task, b_bn))}

def _parse_spec(spec):
    """'digits' | 'digits_mlp:donor' | 'digits:self' ... -> (kind, variant)."""
    kind, _, var = spec.partition(":")
    assert var in ("", "donor", "self"), spec
    return kind, var

def run_bottleneck(arm, train_items, X9tr, X2tr, ref, donor_top_tr):
    """ref = per-item lists on `unsolv`: {'task_full_data', 'recon' (first token), 'task_full'
    (full answer)}. donor_top_tr = the donor's own top-1 token on each training problem."""
    t0 = time.time()
    ridx = list(range(len(train_items))); random.Random(BN_SEED).shuffle(ridx)
    cut = int(len(ridx) * BN_READER_FRAC)
    R_IDX, W_IDX = ridx[:cut], ridx[cut:]
    bin_items = [evalp[j] for j in unsolv]
    # bin items whose answer class the WRITER half never saw: a per-class code can't be trained there
    writer_classes = {train_items[k]["tok"] for k in W_IDX}
    S = [k for k, j in enumerate(unsolv) if evalp[j]["tok"] in writer_classes]
    U = [k for k, j in enumerate(unsolv) if evalp[j]["tok"] not in writer_classes]
    # bin items whose operand-swapped twin (b * a + c, same answer) is a TRAINING problem: the map
    # may have seen a near-identical donor state with the same label, so gains there can be lookup
    _tr_exprs = {p["expr"] for p in train_items}
    TW = [k for k, j in enumerate(unsolv) if _twin(evalp[j]["expr"]) in _tr_exprs]
    NT = [k for k, j in enumerate(unsolv) if _twin(evalp[j]["expr"]) not in _tr_exprs]
    donor_ans = [_answer_of_token(t) for t in donor_top_tr]
    print(f"[{arm}] reader n={len(R_IDX)} | writer n={len(W_IDX)} | bin n={len(unsolv)} | "
          f"bin answers seen by the writer half: {len(S)}/{len(unsolv)} | swap twin in train: {len(TW)} | "
          f"donor right on {sum(donor_ans[i] == train_items[i]['ans'] for i in R_IDX)}/{len(R_IDX)} reader items")

    MATCHED = _train_matched_task(train_items, X9tr, X2tr, W_IDX)
    MATCH_first = _confer_vecfn(MATCHED, unsolv)
    MATCHED_R = _train_matched_task(train_items, X9tr, X2tr, W_IDX, rank=BN_MATCHED_RANK)
    MATCH_R_first = _confer_vecfn(MATCHED_R, unsolv)
    TASK_first, RECON_first = ref["task_full_data"], ref["recon"]
    _r = lambda v: sum(v) / len(v)
    out = {
        "n_train": len(train_items), "n_reader_train": len(R_IDX), "n_writer_train": len(W_IDX),
        "n_eval_bin": len(unsolv),
        "n_answer_classes_in_train": len({p["tok"] for p in train_items}),
        "n_bin_answers_seen_by_writer_half": len(S),
        "n_bin_with_swap_twin_in_train": len(TW),
        "_ref_task_first_MATCHED_data": fmt(wilson_bools(MATCH_first)),
        f"_ref_task_first_MATCHED_data_rank{BN_MATCHED_RANK}": fmt(wilson_bools(MATCH_R_first)),
        "_ref_task_first_FULL_data": fmt(wilson_bools(TASK_first)),
        "_ref_task_full_FULL_data": fmt(wilson_bools(ref["task_full"])),
        "_ref_recon_first": fmt(wilson_bools(RECON_first)),
        "_ref_native_first": "0.000 [by construction]",
        "_ref_matched_task_first_seen_vs_unseen": [_rate_on(MATCH_first, S), _rate_on(MATCH_first, U)],
        "_ref_task_first_twin_vs_no_twin": {"matched": [_rate_on(MATCH_first, TW), _rate_on(MATCH_first, NT)],
                                            "full_data": [_rate_on(TASK_first, TW), _rate_on(TASK_first, NT)]},
        "readers": {},
    }
    PER_ITEM[arm] = {"matched_task_first": MATCH_first, f"matched_rank{BN_MATCHED_RANK}_task_first": MATCH_R_first,
                     "task_full_data_first": TASK_first, "recon_first": RECON_first,
                     "task_full_data_full": ref["task_full"]}
    BN_MODELS[arm] = {}
    # gold / uniform / masked writers depend only on the CODE (per-class one-hot, or three digit
    # one-hots), not on the reader: train them once per arm and reuse (same seed, same data).
    code_cache = {}

    def _code_writers(code, rd):
        if (code, "gold") in code_cache: return
        z_g = (lambda sub: rd.gold([train_items[k] for k in sub]))
        z_u = (lambda sub: rd.uniform(len(sub)))
        Bg = _train_writer(z_g, rd.width, train_items, X2tr, W_IDX, f"{arm}/{code}/gold")
        Bu = _train_writer(z_u, rd.width, train_items, X2tr, W_IDX, f"{arm}/{code}/uniform")
        vg = lambda sub: rd.gold([evalp[j] for j in sub]) @ Bg[0] + Bg[1]
        vu = lambda sub: rd.uniform(len(sub)) @ Bu[0] + Bu[1]
        code_cache[(code, "gold")] = {"BW": Bg, "first": _confer_vecfn(vg, unsolv),
                                      "full": _full_vecfn(vg, unsolv)}
        code_cache[(code, "uniform")] = {"BW": Bu, "first": _confer_vecfn(vu, unsolv)}
        if code == "digits":
            # masked codes: gold digits with ONE head blanked (train and eval). If the recipient
            # still produces the whole number well above ~1/10 of full-gold conferral, it is
            # computing that digit itself from its own prompt.
            for h_i, nm in ((1, "tens"), (2, "ones")):
                Bm = _train_writer(lambda sub, h_i=h_i: _digit_code([train_items[k]["ans"] for k in sub], mask=h_i),
                                   30, train_items, X2tr, W_IDX, f"{arm}/digits/mask_{nm}")
                vm = (lambda sub, h_i=h_i, Bm=Bm:
                      _digit_code([evalp[j]["ans"] for j in sub], mask=h_i) @ Bm[0] + Bm[1])
                code_cache[(code, f"mask_{nm}")] = {"first": _confer_vecfn(vm, unsolv)}

    TOKSET = sorted({p["tok"] for p in train_items})   # EXP26: classes from the whole training set
    for spec in BN_READERS:
        kind, var = _parse_spec(spec)
        XR, XE = (X2tr, X2e) if var == "self" else (X9tr, X9e)
        if var == "donor":
            rows = [i for i in R_IDX if donor_ans[i] is not None]
            labels = [donor_ans[i] for i in rows]
        else:
            rows = R_IDX
            labels = [train_items[i]["tok"] if kind == "token" else train_items[i]["ans"] for i in rows]
        if kind == "token" and var == "donor":
            raise ValueError("token:donor is not supported (donor answers outside TOKSET)")
        rd = Reader(kind, XR[rows], labels, tokset=TOKSET if kind == "token" else None)
        joint, per = rd.correct_on(XE[unsolv], bin_items)
        print(f"[{arm}/{spec}] channel width {rd.width} | reader n={len(rows)} | accuracy on the bin "
              f"{_r(joint):.3f}" + (f" (hundreds/tens/ones {[round(_r(p), 3) for p in per]})" if per else ""))

        code = "token" if kind == "token" else "digits"
        _code_writers(code, rd)
        Bp = _train_writer(lambda sub, rd=rd, XR=XR: rd.scores(XR[sub]), rd.width,
                           train_items, X2tr, W_IDX, f"{arm}/{spec}/pred")
        BN_MODELS[arm][spec] = (rd, Bp)
        Bg = code_cache[(code, "gold")]["BW"]
        v_pred = lambda sub, rd=rd, XE=XE, Bp=Bp: rd.scores(XE[sub]) @ Bp[0] + Bp[1]
        v_hard = lambda sub, rd=rd, XE=XE, Bg=Bg: rd.hard(XE[sub]) @ Bg[0] + Bg[1]
        v_shuf = lambda sub, rd=rd, XE=XE, Bp=Bp: rd.scores(XE[[PARTNER[j] for j in sub]]) @ Bp[0] + Bp[1]

        pred_first = _confer_vecfn(v_pred, unsolv)
        # the reader's decoded argmax through the GOLD writer: if pred >> hard, the soft scores
        # carry more than the decoded answer (runner-up mass, or non-answer structure)
        hard_first = _confer_vecfn(v_hard, unsolv)
        shuf_first = _confer_vecfn(v_shuf, unsolv)
        shuf_other = _emits_which(v_shuf, unsolv, PARTNER)
        pred_full  = _full_vecfn(v_pred, unsolv)
        gold_first = code_cache[(code, "gold")]["first"]; gold_full = code_cache[(code, "gold")]["full"]
        unif_first = code_cache[(code, "uniform")]["first"]

        c_bn, c_match, c_task = _r(pred_first), _r(MATCH_first), _r(TASK_first)
        res = {
            "reader_states": "recipient L11 (self control)" if var == "self" else "donor L17",
            "reader_labels": "donor's own top-1 answer" if var == "donor" else "gold answer",
            "n_reader_train_used": len(rows),
            "channel_width": rd.width, "scores_are_softmax": bool(BN_SOFTMAX),
            "reader_acc_on_bin": fmt(wilson_bools(joint)),
            "bottleneck_pred_first":     fmt(wilson_bools(pred_first)),
            "bottleneck_pred_hard_first": fmt(wilson_bools(hard_first)),
            "paired_pred_vs_hard": dict(zip(("hard_only", "pred_only", "mcnemar_p"), mcnemar(pred_first, hard_first))),
            "bottleneck_gold_first":     fmt(wilson_bools(gold_first)),
            "bottleneck_uniform_first":  fmt(wilson_bools(unif_first)),
            "bottleneck_shuffled_first": fmt(wilson_bools(shuf_first)),
            "bottleneck_shuffled_emits_PARTNER_answer": fmt(wilson_bools(shuf_other)),
            "bottleneck_pred_full":      fmt(wilson_bools(pred_full)),
            "bottleneck_gold_full":      fmt(wilson_bools(gold_full)),
            "recovery_vs_matched_task":  round(c_bn / c_match, 4) if c_match else None,
            "recovery_vs_full_task":     round(c_bn / c_task, 4) if c_task else None,
            "paired_vs_matched_task":    _paired(MATCH_first, pred_first),
            f"paired_vs_matched_rank{BN_MATCHED_RANK}_task": _paired(MATCH_R_first, pred_first),
            "paired_vs_full_task":       _paired(TASK_first, pred_first),
            "pred_first_twin_vs_no_twin": [_rate_on(pred_first, TW), _rate_on(pred_first, NT)],
            "by_writer_half_class_coverage": {
                "_note": ("'seen' = the bin item's answer class occurs in the writer half. For the token "
                          "code, unseen classes fail by construction (untrained rows); the split is "
                          "diagnostic only for the digit code."),
                "n_seen": len(S), "n_unseen": len(U),
                "pred_first_seen": _rate_on(pred_first, S), "pred_first_unseen": _rate_on(pred_first, U),
                "gold_first_seen": _rate_on(gold_first, S), "gold_first_unseen": _rate_on(gold_first, U),
                "matched_task_first_seen": _rate_on(MATCH_first, S),
                "matched_task_first_unseen": _rate_on(MATCH_first, U),
            },
        }
        if per:
            res["reader_acc_per_digit"] = {nm: fmt(wilson_bools(p))
                                           for nm, p in zip(("hundreds", "tens", "ones"), per)}
        if kind == "token":
            iv = [k for k, j in enumerate(unsolv) if evalp[j]["tok"] in rd.tok2cls]
            res["n_eval_answers_outside_train_classes"] = len(unsolv) - len(iv)
            res["bottleneck_gold_first_invocab"] = fmt(wilson_bools([gold_first[k] for k in iv]))
        out["readers"][spec] = res
        PER_ITEM[arm][spec] = {"reader_correct": joint, "pred_first": pred_first, "pred_hard_first": hard_first,
                               "gold_first": gold_first, "uniform_first": unif_first,
                               "shuffled_first": shuf_first, "shuffled_emits_partner": shuf_other,
                               "pred_full": pred_full, "gold_full": gold_full}
        print(f"[{arm}/{spec}] pred {c_bn:.3f} | hard {_r(hard_first):.3f} | gold {_r(gold_first):.3f} | "
              f"uniform {_r(unif_first):.3f} | matched task {c_match:.3f} (rank-{BN_MATCHED_RANK} "
              f"{_r(MATCH_R_first):.3f}) | full task {c_task:.3f}")

    if ("digits", "mask_tens") in code_cache:
        g = code_cache[("digits", "gold")]["first"]
        def _prior_correct(n_mask):
            # does the most common value of the blanked digit, given the other two digits, among the
            # WRITER half's answers equal the item's true digit? A writer can learn this prior, so a
            # masked code can confer without the recipient computing anything.
            tab = collections.defaultdict(collections.Counter)
            for k in W_IDX:
                d = _digits(train_items[k]["ans"])
                tab[tuple(d[i] for i in range(3) if i != n_mask)][d[n_mask]] += 1
            out_ = []
            for j in unsolv:
                d = _digits(evalp[j]["ans"]); key = tuple(d[i] for i in range(3) if i != n_mask)
                out_.append(bool(tab[key]) and tab[key].most_common(1)[0][0] == d[n_mask])
            return out_
        nulls = {nm: [bool(a_ and b_) for a_, b_ in zip(g, _prior_correct(h_i))]
                 for h_i, nm in ((1, "tens"), (2, "ones"))}
        out["masked_digit_code"] = {
            "_what": ("gold digit code with one head blanked to uniform, in training and eval. LOOKUP NULL = "
                      "items gold confers AND whose blanked digit equals the writer half's most common value "
                      "given the other two digits: what writer + recipient get from the answer prior alone. "
                      "Masked conferral clearly above the lookup null means the recipient computes the digit "
                      "from its own prompt."),
            "gold_first": fmt(wilson_bools(g)),
            **{f"mask_{nm}_first": fmt(wilson_bools(code_cache[("digits", f"mask_{nm}")]["first"]))
               for nm in ("tens", "ones")},
            **{f"mask_{nm}_over_gold": (round(_r(code_cache[("digits", f"mask_{nm}")]["first"]) / _r(g), 3)
                                        if _r(g) else None) for nm in ("tens", "ones")},
            **{f"mask_{nm}_LOOKUP_NULL": fmt(wilson_bools(nulls[nm])) for nm in ("tens", "ones")},
            **{f"paired_mask_{nm}_vs_null": dict(zip(("null_only", "mask_only", "mcnemar_p"),
                                                     mcnemar(nulls[nm], code_cache[("digits", f"mask_{nm}")]["first"])))
               for nm in ("tens", "ones")},
        }
        for nm in ("tens", "ones"):
            PER_ITEM[arm][f"digits_mask_{nm}_first"] = code_cache[("digits", f"mask_{nm}")]["first"]
    for ka, kb in (("token", "digits"), ("digits", "digits_mlp"), ("digits", "digits:donor"),
                   ("digits_mlp", "digits_mlp:donor"), ("digits", "digits:self"),
                   ("digits_mlp", "digits_mlp:self")):
        if ka in out["readers"] and kb in out["readers"]:
            n01, n10, p = mcnemar(PER_ITEM[arm][ka]["pred_first"], PER_ITEM[arm][kb]["pred_first"])
            out[f"paired_{ka}_vs_{kb}_pred"] = {f"{ka}_only": n10, f"{kb}_only": n01, "mcnemar_p": p}
    out["_minutes"] = round((time.time() - t0) / 60, 1)
    return out

# the donor's own top-1 token on each of the 3,000 training problems ('...:donor' reader labels)
DONOR_TOP_TRAIN = states_and_top(model_9b, [], [p["ids"] for p in train])[1]
print(f"donor right on {sum(t == p['tok'] for t, p in zip(DONOR_TOP_TRAIN, train))}/{len(train)} training problems; "
      f"its top-1 is a 3-digit number on {sum(_answer_of_token(t) is not None for t in DONOR_TOP_TRAIN)}")

print("bottleneck machinery defined")

shuffled pairing: 626 pairs, 0 unavoidable same-token collisions
donor right on 944/3000 training problems; its top-1 is a 3-digit number on 2933
bottleneck machinery defined


## Part B — bottleneck on the paper's 3,000 training problems
Same bin, same training set as Part A. The full-data reference is Part A's seed-0 task map (the one behind the paper's 0.278).

In [14]:
# === CELL B1: Part B ===
if RUN_BASE_BOTTLENECK:
    REF_BASE = {"task_full_data": first_token_confer(map_task(0), unsolv),
                "recon": first_token_confer(map_recon, unsolv),
                "task_full": full_confer(map_task(0), unsolv)}   # EXP26's _ref_task_full
    RESULTS["bottleneck_base"] = run_bottleneck("base", train, X9t, X2t, REF_BASE, DONOR_TOP_TRAIN)
    print(json.dumps(RESULTS["bottleneck_base"], indent=2))
    save_partial("B")
else:
    print("Part B skipped")

[base] reader n=1500 | writer n=1500 | bin n=626 | bin answers seen by the writer half: 551/626 | swap twin in train: 81 | donor right on 458/1500 reader items
[base/token] channel width 695 | reader n=1500 | accuracy on the bin 0.021
     writer[base/token/gold] epoch 0  CE = 5.0099
     writer[base/token/gold] epoch 4  CE = 0.0032
     writer[base/token/gold] epoch 7  CE = 0.0013
     writer[base/token/uniform] epoch 0  CE = 5.7578
     writer[base/token/uniform] epoch 4  CE = 4.1114
     writer[base/token/uniform] epoch 7  CE = 3.6077
     writer[base/token/pred] epoch 0  CE = 4.9303
     writer[base/token/pred] epoch 4  CE = 0.7554
     writer[base/token/pred] epoch 7  CE = 0.0991
[base/token] pred 0.078 | hard 0.024 | gold 0.864 | uniform 0.168 | matched task 0.198 (rank-27 0.521) | full task 0.278
[base/digits] channel width 30 | reader n=1500 | accuracy on the bin 0.094 (hundreds/tens/ones [0.885, 0.179, 0.545])
     writer[base/digits/gold] epoch 0  CE = 1.7805
     writer[base

## Part C — the same test at 18,000 training problems
The 3,000 problems from Part A are the first 3,000 here (same generator, same seed); every one of the 2,000 generated eval expressions is excluded. A new full-data task map (EXP3 recipe, seed 0) and reconstruction map are fit on all 18,000 for reference. The paper's data-scale numbers (0.217 → 0.586, App. A.18) come from a different run (EXP28: another 18k draw, two epochs, only the 642 donor-solved eval items excluded), so the comparison that matters is within this cell.

In [15]:
# === CELL C1: Part C ===
def train_task_map(train_items, X9tr, recon, seed, epochs):
    """EXP3 recipe (A2), one seed, on an arbitrary training set."""
    _mu9, _mu2, _Wr = recon
    _mu9d = _mu9.to(DEVICE)
    torch.manual_seed(seed); random.seed(seed)
    W = _Wr.clone().to(DEVICE).requires_grad_(True); b = _mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    model_2b.requires_grad_(False)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train_items)))
    try:
        for ep in range(epochs):
            random.Random(seed*100 + ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s + ARITH_BATCH]
                _graft["vec"] = (X9tr[sub].to(DEVICE) - _mu9d) @ W + b
                ids, m = left_pad([train_items[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train_items[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
            print(f"  scaled task map epoch {ep}: last batch CE {loss.item():.3f}")
    finally:
        handle.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
    W, b = W.detach(), b.detach()
    return lambda x9: (x9.to(DEVICE) - _mu9d) @ W + b      # a function of donor states

TASKBIG = None
if RUN_SCALED_BOTTLENECK:
    t0 = time.time()
    train_big = gen_arith(tokenizer, N_ARITH_TRAIN_SCALED, random.Random(0), EVAL_EXPRS_ALL)
    leak = sum(p["expr"] in EVAL_EXPRS_ALL for p in train_big)
    nested = [p["expr"] for p in train_big[:len(train)]] == [p["expr"] for p in train]
    print(f"scaled train n={len(train_big)} | eval expressions inside it: {leak} (want 0) | "
          f"Part A set nested as its first {len(train)}: {nested}")
    assert leak == 0
    if nested:   # reuse Part A's states, capture only the new problems
        _new = [p["ids"] for p in train_big[len(train):]]
        _x9, _top_new = states_and_top(model_9b, [L9_SINGLE], _new)
        _x2, _ = states_and_top(model_2b, [L2_SINGLE], _new)
        X9b = torch.cat([X9t, _x9[L9_SINGLE]]); X2b = torch.cat([X2t, _x2[L2_SINGLE]])
        DONOR_TOP_BIG = list(DONOR_TOP_TRAIN) + list(_top_new)
    else:
        _all = [p["ids"] for p in train_big]
        _x9, DONOR_TOP_BIG = states_and_top(model_9b, [L9_SINGLE], _all); X9b = _x9[L9_SINGLE]
        X2b = states_and_top(model_2b, [L2_SINGLE], _all)[0][L2_SINGLE]
    print(f"answer classes in scaled train: {len({p['tok'] for p in train_big})}")

    recon_big = fit_ridge(X9b, X2b)
    _m9, _m2, _Wb = (t.to(DEVICE) for t in recon_big)
    RECONBIG_first = _confer_vecfn(lambda sub: (X9e[sub].to(DEVICE) - _m9) @ _Wb + _m2, unsolv)
    TASKBIG = train_task_map(train_big, X9b, recon_big, seed=0, epochs=TASK_EPOCHS)
    TASKBIG_first = _confer_vecfn(lambda sub: TASKBIG(X9e[sub]), unsolv)
    n01, n10, p = mcnemar(REF_BASE["task_full_data"] if RUN_BASE_BOTTLENECK
                          else first_token_confer(map_task(0), unsolv), TASKBIG_first)
    RESULTS["scaled_reference_maps"] = {
        "n_train": len(train_big),
        "task_first_18k": fmt(wilson_bools(TASKBIG_first)),
        "recon_first_18k": fmt(wilson_bools(RECONBIG_first)),
        "task_first_3k_seed0_partA": RESULTS["arithmetic"]["task_unsolv_first_pooled"],
        "paired_3k_vs_18k_task": {"3k_only": n10, "18k_only": n01, "mcnemar_p": p},
        "_note": "paper's data-scale appendix: 0.217 -> 0.586 under a shorter schedule",
    }
    print(json.dumps(RESULTS["scaled_reference_maps"], indent=2))
    RESULTS["bottleneck_scaled"] = run_bottleneck(
        "scaled", train_big, X9b, X2b, {"task_full_data": TASKBIG_first, "recon": RECONBIG_first,
                                        "task_full": _full_vecfn(lambda s: TASKBIG(X9e[s]), unsolv)},
        DONOR_TOP_BIG)
    RESULTS["bottleneck_scaled"]["_minutes_incl_states_and_ref_maps"] = round((time.time() - t0) / 60, 1)
    save_partial("C")
    print(json.dumps(RESULTS["bottleneck_scaled"], indent=2))
else:
    print("Part C skipped")

scaled train n=18000 | eval expressions inside it: 0 (want 0) | Part A set nested as its first 3000: True
answer classes in scaled train: 825
  scaled task map epoch 0: last batch CE 2.220
  scaled task map epoch 1: last batch CE 0.991
  scaled task map epoch 2: last batch CE 1.286
  scaled task map epoch 3: last batch CE 0.782
  scaled task map epoch 4: last batch CE 0.585
  scaled task map epoch 5: last batch CE 1.008
{
  "n_train": 18000,
  "task_first_18k": "0.746 [0.710, 0.779]",
  "recon_first_18k": "0.016 [0.009, 0.029]",
  "task_first_3k_seed0_partA": "0.278 [0.244, 0.314]",
  "paired_3k_vs_18k_task": {
    "3k_only": 23,
    "18k_only": 316,
    "mcnemar_p": 5.432646901965075e-67
  },
  "_note": "paper's data-scale appendix: 0.217 -> 0.586 under a shorter schedule"
}
[scaled] reader n=9000 | writer n=9000 | bin n=626 | bin answers seen by the writer half: 623/626 | swap twin in train: 409 | donor right on 2841/9000 reader items
[scaled/token] channel width 825 | reader n=9000 

## Part D — prompt-mismatch test
Llama port of Experiment A in `Subspace-ablation-followups/48GB_AnswerSubspace_followups` (the paper's §6.2 perturbation test). The recipient reads P′ = (a, b, c + δ) while the graft comes from the donor's state on P = (a, b, c). If the recipient finishes a partial answer from the operands on its own screen, the digits the graft carries weakly will follow P′'s answer y′ instead of P's answer y.

Two δ rules, run separately:
- **`leading`** — Experiment A's own rule: the smallest |δ| that changes the leading digit. STALE/ADAPTED are also scored on the leading digit, which is the quantity Gemma's first token measured (paper: 0.890 / 0.088, native 0.366, matched control 0.476).
- **`far`** — for digit-by-digit scoring: δ drawn at random (seeded) among shifts that change the ones digit, move the tens digit by at least 2, and keep y′ off a multiple of ten. The smallest-|δ| rule would put y′ next to y (median |δ| 3; y′ is y rounded to the nearest ten about half the time), so a graft that merely lands *near* y would look like it follows the prompt.

In [16]:
# === CELL D1: Part D — prompt-mismatch test ===
#   P  = (a, b, c)          -> donor state on P            (= X9e[i], i in the bin)
#   P' = (a, b, c + delta)  -> the RECIPIENT reads P' and gets f(donor state) grafted
# Conditions per map f (recipient reads P' unless noted):
#   MISMATCH : f(donor on P)
#   MATCHED  : f(donor on P')          control: can the pipeline emit y' at all?
#   SHUFFLED : f(donor on unrelated U) positive control (does it emit U's answer?) and a read of how
#                                      much the recipient computes from P' when the graft says nothing
#                                      about P'. NOT a chance baseline: recipient completion would
#                                      raise its y'-agreement too.
#   NATIVE   : no graft                what the recipient computes alone on P'
#   ORIGINAL : f(donor on P), recipient reads P. Scored against y it is the ordinary conferral on
#              these items; scored against y' it is the PROXIMITY NULL — how often the graft alone
#              lands on y''s digits with P' nowhere on screen.
#   CHANCE   : MISMATCH outputs scored against a third, unrelated problem's answer, 20 draws (the
#              paper's empirical null, app:shuffle).
# A digit is "completed by the recipient" when MISMATCH matches y' there clearly more often than
# both ORIGINAL-vs-y' and CHANCE.
# Differences from Experiment A: the 'far' rule and per-digit scoring are additions; c' stays in
# Llama's c range [1, 40] and y' in [100, 999]; prompt length must match exactly (A recorded it);
# the SHUFFLED donor's answer must differ from y and y' (A required only a different item); A's
# full-answer classification, neither-bucket digit check and examples are not ported (the paper's
# §6.2 numbers are first-token only).
import collections, re

DIGIT_NAMES = ("hundreds", "tens", "ones")
CHANCE_DRAWS = 20

@torch.inference_mode()
def _tops(items, V):
    # first-token argmax on each item's prompt, with V[k] grafted (no graft if V is None)
    h = (model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
         if V is not None else None)
    out = []
    try:
        for i in range(0, len(items), ARITH_BATCH):
            chunk = items[i:i + ARITH_BATCH]
            _graft["vec"] = V[i:i + len(chunk)].to(DEVICE) if V is not None else None
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            out += model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
    finally:
        if h is not None: h.remove()
        _graft["vec"] = None
    return out

def _agree(tops, items, ks, ans_of):
    """Per item in ks: does the output agree with answer ans_of(k) — whole number, leading
    character (Gemma's first-token metric), and each place-value digit."""
    out = {"whole": [], "lead": [], **{nm: [] for nm in DIGIT_NAMES}}
    for k in ks:
        s = tokenizer.decode([tops[k]]).strip(); A = ans_of(k); sA = str(A); dA = _digits(A)
        out["whole"].append(s == sA)
        out["lead"].append(s[:1] == sA[0])
        d3 = [int(ch) for ch in s] if (len(s) == 3 and s.isdigit()) else None
        for n, nm in enumerate(DIGIT_NAMES):
            out[nm].append(d3 is not None and d3[n] == dA[n])
    return out

def _differs(items, k, n): return _digits(items[k]["y"])[n] != _digits(items[k]["y2"])[n]

def _rates(tops, items, ks, ans_of):
    """Mean agreement; place-value digits only over items where y and y' differ at that digit."""
    a = _agree(tops, items, ks, ans_of); r = {"whole": a["whole"], "lead": a["lead"]}
    for n, nm in enumerate(DIGIT_NAMES):
        r[nm] = [v for v, k in zip(a[nm], ks) if _differs(items, k, n)]
    return r

def _fmt_rates(r):
    return {key: (fmt(wilson_bools(v)) if len(v) else None) for key, v in r.items()}

def _mean_rates(r):
    return {key: (float(np.mean(v)) if len(v) else float("nan")) for key, v in r.items()}

def _build_items(rule, seed):
    rng = random.Random(seed)
    items, drop = [], 0
    for i in unsolv:
        mm = _EXPR_RE.match(evalp[i]["expr"])
        if mm is None: drop += 1; continue
        a, b, c = map(int, mm.groups()); y = a * b + c
        assert y == evalp[i]["ans"], "expr/ans mismatch"
        dy = _digits(y); cands = []
        for dlt in DELTAS:                                   # ordered by |delta|
            c2 = c + dlt; y2 = a * b + c2
            if not (1 <= c2 <= ARITH_C_HI and ARITH_ANS_MIN <= y2 <= ARITH_ANS_MAX): continue
            if _ntok(c2) != _ntok(c): continue
            d2 = _digits(y2)
            if rule == "leading":
                if d2[0] == dy[0]: continue                   # Experiment A: leading digit flips
            elif rule == "far":
                if d2[2] == dy[2] or abs(d2[1] - dy[1]) < 2 or y2 % 10 == 0: continue
            else:
                raise ValueError(rule)
            cands.append((dlt, c2, y2))
        if rule == "far": rng.shuffle(cands)                 # 'leading' keeps smallest |delta| first
        pick = None
        for dlt, c2, y2 in cands:
            ids2, tok2 = _aencode(tokenizer, f"{a} * {b} + {c2}", y2)
            if ids2 is None or tok2 == evalp[i]["tok"]: continue
            if ids2.numel() != evalp[i]["ids"].numel(): continue    # graft position unchanged
            pick = dict(idx=i, a=a, b=b, c=c, c2=c2, delta=dlt, y=y, y2=y2, ans=y2,
                        expr=f"{a} * {b} + {c2}", ids=ids2, tok=tok2, stale_tok=evalp[i]["tok"])
            break
        if pick is None: drop += 1
        else: items.append(pick)
    return items, drop

def run_perturb(rule, MAPS):
    items, drop = _build_items(rule, P_SHUFFLE_SEED)
    n = len(items)
    print(f"[D/{rule}] kept {n}/{len(unsolv)} bin problems ({drop} have no valid delta)")
    if n == 0: return {"rule": rule, "n_with_valid_delta": 0, "n_dropped_no_delta": drop}
    src = [p["idx"] for p in items]
    _x9p, donor_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in items])
    X9P = _x9p[L9_SINGLE]                                            # donor on P'
    rng = np.random.default_rng(P_SHUFFLE_SEED); shuf_src = []
    for k in range(n):
        while True:
            s_ = int(rng.integers(0, len(evalp)))
            if s_ != src[k] and evalp[s_]["ans"] not in (items[k]["y"], items[k]["y2"]): break
        shuf_src.append(s_)
    # third-problem draws for the CHANCE null (answers differ from y and y')
    third = []
    for _ in range(CHANCE_DRAWS):
        row = []
        for k in range(n):
            while True:
                u = int(rng.integers(0, len(evalp)))
                if evalp[u]["ans"] not in (items[k]["y"], items[k]["y2"]): break
            row.append(evalp[u]["ans"])
        third.append(row)
    orig_items = [evalp[i] for i in src]

    top_native = _tops(items, None)
    # P' must ALSO be natively unsolvable, else ADAPTED is contaminated by native ability (as A)
    KS = [k for k in range(n) if top_native[k] != items[k]["tok"]]
    KS_DS = [k for k in KS if donor_top[k] == items[k]["tok"]]          # donor solves P'
    # P' can be a TRAINING problem (or its operand-swap twin). Under MATCHED the map is then fed a
    # donor state it was trained to send to y', so MATCHED is also reported on P' outside the map's
    # own training set. MISMATCH / SHUFFLED / NATIVE / ORIGINAL never feed the donor state on P'.
    _seen3 = {p["expr"] for p in train}
    _seen18 = {p["expr"] for p in train_big} if TASKBIG is not None else set()
    def _in(seen, k):
        e = items[k]["expr"]; return e in seen or _twin(e) in seen
    KS_OOT = {"3k": [k for k in KS if not _in(_seen3, k)],
              "18k": [k for k in KS if not _in(_seen3 | _seen18, k)]}
    print(f"[D/{rule}] P' also natively unsolvable: {len(KS)}/{n} | donor solves P' on {len(KS_DS)}")
    y_of  = lambda k: items[k]["y"]
    y2_of = lambda k: items[k]["y2"]
    u_of  = lambda k: evalp[shuf_src[k]]["ans"]

    out = {"rule": rule, "n_with_valid_delta": n, "n_dropped_no_delta": drop,
           "n_scored_Pprime_also_unsolvable": len(KS), "n_donor_solves_Pprime": len(KS_DS),
           "n_Pprime_or_twin_in_train3k": len(KS) - len(KS_OOT["3k"]),
           "n_Pprime_or_twin_in_train18k": (len(KS) - len(KS_OOT["18k"])) if TASKBIG is not None else None,
           "abs_delta_median": float(np.median([abs(items[k]["delta"]) for k in KS])) if KS else None,
           "n_digit_differs": {nm: sum(_differs(items, k, i_) for k in KS)
                               for i_, nm in enumerate(DIGIT_NAMES)},
           "NATIVE": {"matches_y": _fmt_rates(_rates(top_native, items, KS, y_of)),
                      "matches_y2": _fmt_rates(_rates(top_native, items, KS, y2_of))},
           "maps": {}}
    seeds = collections.defaultdict(list)
    for name, f in MAPS.items():
        is_seed = name.startswith("task_3k_seed")
        top_mis = _tops(items, f(X9e[src]))
        top_mat = _tops(items, f(X9P))
        r_mis_y, r_mis_y2 = _rates(top_mis, items, KS, y_of), _rates(top_mis, items, KS, y2_of)
        r_mat_y2 = _rates(top_mat, items, KS, y2_of)
        _oot = KS_OOT["18k" if name.endswith("18k") else "3k"]
        row = {"MISMATCH": {"matches_y": _fmt_rates(r_mis_y), "matches_y2": _fmt_rates(r_mis_y2),
                            "NEITHER_whole": fmt(wilson_bools([not (a_ or b_) for a_, b_ in
                                                               zip(r_mis_y["whole"], r_mis_y2["whole"])]))},
               "MISMATCH_Pprime_not_in_this_maps_train": {
                   "n": len(_oot), "matches_y": _fmt_rates(_rates(top_mis, items, _oot, y_of)),
                   "matches_y2": _fmt_rates(_rates(top_mis, items, _oot, y2_of))},
               "MATCHED": {"matches_y2": _fmt_rates(r_mat_y2),
                           "matches_y2_where_donor_solves_Pprime":
                               _fmt_rates(_rates(top_mat, items, KS_DS, y2_of)),
                           "matches_y2_Pprime_not_in_this_maps_train":
                               _fmt_rates(_rates(top_mat, items, _oot, y2_of))}}
        if is_seed:
            for tag, r in (("MISMATCH_y", r_mis_y), ("MISMATCH_y2", r_mis_y2), ("MATCHED_y2", r_mat_y2)):
                for key, v in _mean_rates(r).items(): seeds[f"{tag}.{key}"].append(v)
        if not is_seed or name == "task_3k_seed0":
            top_sh = _tops(items, f(X9e[shuf_src]))
            top_o  = _tops(orig_items, f(X9e[src]))
            row["SHUFFLED"] = {"matches_U_donor": _fmt_rates(_rates(top_sh, items, KS, u_of)),
                               "matches_y2": _fmt_rates(_rates(top_sh, items, KS, y2_of))}
            row["ORIGINAL"] = {"matches_y_conferral": _fmt_rates(_rates(top_o, items, KS, y_of)),
                               "matches_y2_PROXIMITY_NULL": _fmt_rates(_rates(top_o, items, KS, y2_of))}
            ch = collections.defaultdict(list)
            for d in range(CHANCE_DRAWS):
                for key, v in _mean_rates(_rates(top_mis, items, KS, lambda k, d=d: third[d][k])).items():
                    ch[key].append(v)
            row["CHANCE_null_MISMATCH_outputs"] = {key: f"{np.nanmean(v):.3f} (SD {np.nanstd(v):.3f})"
                                                   for key, v in ch.items()}
            if name == "task_3k_seed0":
                out["per_item_task_3k_seed0"] = {
                    "exprs_Pprime": [items[k]["expr"] for k in KS],
                    "stale_whole": [int(x) for x in r_mis_y["whole"]],
                    "adapted_whole": [int(x) for x in r_mis_y2["whole"]]}
        out["maps"][name] = row
        print(f"  [D/{rule}] {name:26s} MISMATCH lead y {_fmt_rates(r_mis_y)['lead']} / y' "
              f"{_fmt_rates(r_mis_y2)['lead']} | whole y {_fmt_rates(r_mis_y)['whole']} / y' "
              f"{_fmt_rates(r_mis_y2)['whole']}")
    if seeds:
        out["task_3k_across_seeds"] = {key: fmt(across_seed_ci([x for x in v if not math.isnan(x)]))
                                       for key, v in seeds.items()
                                       if any(not math.isnan(x) for x in v)}
    return out

if RUN_PERTURB and len(unsolv):
    t0 = time.time()
    _EXPR_RE = re.compile(r"^(\d+) \* (\d+) \+ (\d+)$")
    DELTAS = [d for k in range(1, ARITH_C_HI) for d in (k, -k)]      # smallest |delta| first
    _NT = {}
    def _ntok(x):
        if x not in _NT: _NT[x] = len(tokenizer(str(x), add_special_tokens=False).input_ids)
        return _NT[x]

    MAPS = {f"task_3k_seed{sd}": (lambda x9, W=W, b=b: (x9.to(DEVICE) - mu9d) @ W + b)
            for sd, (W, b) in enumerate(task_maps)}
    for arm_name, tag in (("base", "3k"), ("scaled", "18k")):
        for spec, (rd, (Bw, bw)) in BN_MODELS.get(arm_name, {}).items():
            if spec.startswith("digits") and not spec.endswith(":self"):   # donor-side digit readers
                MAPS[f"bottleneck_{spec}_{tag}"] = (lambda x9, rd=rd, Bw=Bw, bw=bw: rd.scores(x9) @ Bw + bw)
    if TASKBIG is not None: MAPS["task_18k"] = TASKBIG

    RESULTS["perturbation"] = {
        "_what": ("recipient reads P'=(a,b,c+delta); graft = f(donor on X). MISMATCH X=P, MATCHED X=P', "
                  "SHUFFLED X=unrelated U, NATIVE no graft; ORIGINAL = recipient reads P with X=P "
                  "(vs y: conferral; vs y': proximity null); CHANCE = MISMATCH outputs vs a third "
                  "problem's answer, 20 draws. 'lead' = first character of the output token, Gemma's "
                  "first-token metric; place-value digits counted only where y and y' differ."),
        "rules": {rule: run_perturb(rule, MAPS) for rule in P_RULES},
        "_minutes": round((time.time() - t0) / 60, 1),
    }
    save_partial("D")
    print(json.dumps({r: {k: v for k, v in d.items() if k != "per_item_task_3k_seed0"}
                      for r, d in RESULTS["perturbation"]["rules"].items()}, indent=1)[:20000])
else:
    print("Part D skipped")

[D/leading] kept 179/626 bin problems (447 have no valid delta)
[D/leading] P' also natively unsolvable: 159/179 | donor solves P' on 53
  [D/leading] task_3k_seed0              MISMATCH lead y 0.704 [0.629, 0.770] / y' 0.296 [0.230, 0.371] | whole y 0.182 [0.130, 0.250] / y' 0.025 [0.010, 0.063]
  [D/leading] task_3k_seed1              MISMATCH lead y 0.774 [0.703, 0.832] / y' 0.226 [0.168, 0.297] | whole y 0.233 [0.174, 0.304] / y' 0.050 [0.026, 0.096]
  [D/leading] task_3k_seed2              MISMATCH lead y 0.774 [0.703, 0.832] / y' 0.226 [0.168, 0.297] | whole y 0.239 [0.179, 0.311] / y' 0.031 [0.014, 0.071]
  [D/leading] task_3k_seed3              MISMATCH lead y 0.774 [0.703, 0.832] / y' 0.220 [0.163, 0.291] | whole y 0.233 [0.174, 0.304] / y' 0.019 [0.006, 0.054]
  [D/leading] task_3k_seed4              MISMATCH lead y 0.736 [0.662, 0.798] / y' 0.258 [0.196, 0.331] | whole y 0.170 [0.119, 0.236] / y' 0.031 [0.014, 0.071]
  [D/leading] bottleneck_digits_3k       MISMATCH lead y 0

## Summary and save

In [17]:
# === CELL S1: summary table + save ===
def _row(arm):
    a = RESULTS.get(f"bottleneck_{arm}")
    if not a: return
    print(f"\n{arm.upper()}  (train n={a['n_train']}, reader/writer {a['n_reader_train']}/{a['n_writer_train']}, "
          f"bin n={a['n_eval_bin']}, answer classes {a['n_answer_classes_in_train']})")
    print(f"  task map, full data   first {a['_ref_task_first_FULL_data']}   full {a['_ref_task_full_FULL_data']}")
    print(f"  task map, matched     first {a['_ref_task_first_MATCHED_data']}   "
          f"rank {BN_MATCHED_RANK}: {a[f'_ref_task_first_MATCHED_data_rank{BN_MATCHED_RANK}']}")
    print(f"  reconstruction map    first {a['_ref_recon_first']}")
    print(f"  bin items whose operand-swap twin is a training problem: {a['n_bin_with_swap_twin_in_train']}  "
          f"(task first, twin / no twin: {a['_ref_task_first_twin_vs_no_twin']})")
    for kind, r in a["readers"].items():
        pm, pf, ph = r["paired_vs_matched_task"], r["paired_vs_full_task"], r["paired_pred_vs_hard"]
        print(f"  [{kind}]  {r['reader_states']}, {r['reader_labels']}, n={r['n_reader_train_used']}, "
              f"channel width {r['channel_width']}")
        print(f"      reader accuracy (ceiling)  {r['reader_acc_on_bin']}"
              + (f"   per digit {r['reader_acc_per_digit']}" if "reader_acc_per_digit" in r else ""))
        print(f"      predicted scores           {r['bottleneck_pred_first']}   full {r['bottleneck_pred_full']}   "
              f"twin-in-train / not {r['pred_first_twin_vs_no_twin']}")
        print(f"          recovery vs matched {r['recovery_vs_matched_task']} (task-only {pm['task_only']} vs "
              f"bottleneck-only {pm['bottleneck_only']}, p={pm['mcnemar_p']:.3g})")
        print(f"          recovery vs full    {r['recovery_vs_full_task']} (task-only {pf['task_only']} vs "
              f"bottleneck-only {pf['bottleneck_only']}, p={pf['mcnemar_p']:.3g})")
        print(f"          minus task: matched {pm['diff_bottleneck_minus_task']} | full {pf['diff_bottleneck_minus_task']} | "
              f"rank {BN_MATCHED_RANK} {r[f'paired_vs_matched_rank{BN_MATCHED_RANK}_task']['diff_bottleneck_minus_task']}")
        print(f"      reader argmax only         {r['bottleneck_pred_hard_first']}   "
              f"(pred-only {ph['pred_only']} vs hard-only {ph['hard_only']}, p={ph['mcnemar_p']:.3g})")
        print(f"      gold one-hot               {r['bottleneck_gold_first']}")
        print(f"      uniform (bias only)        {r['bottleneck_uniform_first']}")
        print(f"      shuffled donor             {r['bottleneck_shuffled_first']}   "
              f"emits partner's answer {r['bottleneck_shuffled_emits_PARTNER_answer']}")
    if "masked_digit_code" in a:
        mc = a["masked_digit_code"]
        print(f"  masked digit code: gold {mc['gold_first']}")
        for nm in ("tens", "ones"):
            pn = mc[f"paired_mask_{nm}_vs_null"]
            print(f"      {nm} blanked {mc[f'mask_{nm}_first']}  vs lookup null {mc[f'mask_{nm}_LOOKUP_NULL']}  "
                  f"(mask-only {pn['mask_only']} vs null-only {pn['null_only']}, p={pn['mcnemar_p']:.3g})")
for arm in ("base", "scaled"): _row(arm)

P = RESULTS.get("perturbation")
if P:
    for rule, d in P["rules"].items():
        if not d.get("n_scored_Pprime_also_unsolvable"):
            print(f"\nPART D / {rule}: nothing to score"); continue
        keys = ("whole", "lead") if rule == "leading" else ("whole", "tens", "ones")
        print(f"\nPART D / {rule}  (scored n={d['n_scored_Pprime_also_unsolvable']}, median |delta| "
              f"{d['abs_delta_median']}, donor solves P' on {d['n_donor_solves_Pprime']}, P' or its twin "
              f"in training: {d['n_Pprime_or_twin_in_train3k']} (3k) / {d['n_Pprime_or_twin_in_train18k']} (18k))")
        for k in keys:
            print(f"  NATIVE {k:5s}  vs y {d['NATIVE']['matches_y'][k]}   vs y' {d['NATIVE']['matches_y2'][k]}")
        for name, row in d["maps"].items():
            if "ORIGINAL" not in row: continue
            print(f"  [{name}]")
            for k in keys:
                print(f"     {k:5s} ORIGINAL vs y {row['ORIGINAL']['matches_y_conferral'][k]} | "
                      f"MISMATCH vs y {row['MISMATCH']['matches_y'][k]}, vs y' {row['MISMATCH']['matches_y2'][k]} | "
                      f"proximity null {row['ORIGINAL']['matches_y2_PROXIMITY_NULL'][k]} | "
                      f"chance {row['CHANCE_null_MISMATCH_outputs'][k]}")
                print(f"           SHUFFLED vs its donor {row['SHUFFLED']['matches_U_donor'][k]}, vs y' "
                      f"{row['SHUFFLED']['matches_y2'][k]} | MATCHED vs y' {row['MATCHED']['matches_y2'][k]} "
                      f"(P' outside this map's training set: {row['MATCHED']['matches_y2_Pprime_not_in_this_maps_train'][k]})")
        if "task_3k_across_seeds" in d:
            print("  task_3k across seeds:", {kk: v for kk, v in d["task_3k_across_seeds"].items()
                                              if kk.split(".")[1] in keys})

RESULTS["config"] = {
    "pair": f"{MODEL_9B} -> {MODEL_2B}", "graft": {"recipient": L2_SINGLE, "donor": L9_SINGLE},
    "smoke_test": SMOKE_TEST, "task_seeds": TASK_SEEDS, "task_epochs": TASK_EPOCHS,
    "n_train_base": N_ARITH_TRAIN, "n_train_scaled": N_ARITH_TRAIN_SCALED, "n_eval_generated": N_ARITH_EVAL,
    "arith_range": {"a": [2, ARITH_A_HI], "b": [2, ARITH_B_HI], "c": [1, ARITH_C_HI],
                    "ans": [ARITH_ANS_MIN, ARITH_ANS_MAX]},
    "bottleneck": {"readers": list(BN_READERS), "reader_frac": BN_READER_FRAC,
                   "reader_steps": BN_READER_STEPS, "reader_lr": BN_READER_LR,
                   "writer_epochs": BN_WRITER_EPOCHS, "writer_lr": BN_WRITER_LR,
                   "softmax": BN_SOFTMAX, "seed": BN_SEED},
}
RESULTS["per_item_on_bin"] = {
    "bin_eval_exprs": [evalp[j]["expr"] for j in unsolv],
    "arms": {arm: {k: ([int(x) for x in v] if isinstance(v, list) else
                       {kk: [int(x) for x in vv] for kk, vv in v.items()})
                   for k, v in d.items()} for arm, d in PER_ITEM.items()},
}
fname = "llama_bottleneck_results" + ("_SMOKE" if SMOKE_TEST else "") + ".json"
with open(fname, "w") as fh: json.dump(RESULTS, fh, indent=2)
print(f"\nsaved {fname}")


BASE  (train n=3000, reader/writer 1500/1500, bin n=626, answer classes 695)
  task map, full data   first 0.278 [0.244, 0.314]   full 0.278 [0.244, 0.314]
  task map, matched     first 0.198 [0.169, 0.231]   rank 27: 0.521 [0.482, 0.560]
  reconstruction map    first 0.019 [0.011, 0.033]
  bin items whose operand-swap twin is a training problem: 81  (task first, twin / no twin: {'matched': [0.358, 0.174], 'full_data': [0.568, 0.235]})
  [token]  donor L17, gold answer, n=1500, channel width 695
      reader accuracy (ceiling)  0.021 [0.012, 0.035]
      predicted scores           0.078 [0.060, 0.102]   full 0.078 [0.060, 0.102]   twin-in-train / not [0.111, 0.073]
          recovery vs matched 0.3952 (task-only 112 vs bottleneck-only 37, p=5.79e-10)
          recovery vs full    0.2816 (task-only 156 vs bottleneck-only 31, p=2.96e-21)
          minus task: matched -0.120 [-0.157, -0.081] | full -0.200 [-0.240, -0.160] | rank 27 -0.442 [-0.484, -0.399]
      reader argmax only        

## Reading the result

Read in this order. Everything new is single-seed.

**1. Part A.** A4 must pass (it stops the run otherwise). 0.278 is the seed-0 map; A4 also prints the five-seed mean.

**2. Is the comparator meaningful?** The bottleneck *system* uses both halves' labels (reader and writer); the matched map uses one. On saturated Gemma that didn't matter; on data-limited Llama the bottleneck can beat the matched map on data efficiency alone, and in Part B the matched map may sit near floor (if so, read Part C only). So treat the **full-data map** (same total labels) as co-primary, read the **paired differences** (bottleneck − task, bootstrap 95% CI) rather than the recovery ratio, and use the **rank-27 map** as the comparator at the digit channel's width (the paper's Llama low-rank sweep: rank 16 → 0.139, rank 32 → 0.257, full 0.278).

**3. Bottleneck.** Start with `digits:donor` and `digits_mlp:donor` (labels = the donor's own answer), in Part C.

| What you see | What it means |
|---|---|
| `:donor` pred ≈ full and matched task maps (difference CIs include 0), gold high, uniform and shuffled low, partner emission ≈ pred, `:self` near the recon floor (0.019), masked codes near their lookup null | The donor's answer accounts for the task map's conferral on Llama. This answers the objection, provided it holds in Part C where the map is far from saturated. |
| The bottleneck tracks the task map from Part B to Part C, rising with it | The stronger version: as data raises conferral, the donor's answer keeps accounting for it. |
| Pred ≈ reader accuracy, both below the task maps, gold high | The writer is fine; the reader is the limit. If Part C closes the gap, it was data. |
| Gold-label `digits` beats `digits:donor` | The gold-label reader may be computing answers from operand features on problems the donor got wrong. Trust the `:donor` row. |
| `digits_mlp:donor` closes a gap `digits:donor` leaves | The donor's answer is present but nonlinearly encoded; still answer content. (The paper's MLP advantage is on the hundreds digit only: the tens digit reads 0.153 under both probes.) |
| `:self` confers well above the recon floor | A reader + writer can manufacture conferral from what the recipient already has, so donor-side rows must be read against the `:self` rows, not against zero. |
| Pred well above reader accuracy, or pred ≫ the reader's argmax alone | The soft scores carry more than the decoded answer: runner-up mass over nearby numbers, or partial information the recipient completes. Check the masked codes and Part D before calling it answer content. |
| Partner emission well below pred | The recipient's own prompt co-determines the output, a completion signature. |
| A masked code (tens or ones blanked) confers clearly above its lookup null (the writer half's most common value of that digit given the other two) | The recipient computes that digit from its own prompt. **By the paper's §1 definition this is computation transfer** and counts against "answers, not computation" on Llama. |
| Gold low | The additive digit writer can't map three digit codes to the whole-number token through the recipient, so the digit bottleneck is uninformative. (For the token code, failures on unseen classes are by construction.) |
| Token beats digits in Part C | The additive digit code, not the content, is the limit. |
| Pred above the matched map | Fits the paper's examples-per-class account (App. A.18), not "the map carries nothing more". |

**4. Part D.** Read `task_3k_seed0` (the map behind 0.278) against the `:donor` digit bottlenecks.

| What you see | What it means |
|---|---|
| `leading`: MISMATCH lead-vs-y ≈ ORIGINAL lead-vs-y, lead-vs-y′ near NATIVE and chance | The §6.2 result reproduces on Llama: the recipient emits the donor's answer, not its prompt's. Report beside Gemma's 0.890 / 0.088, but don't compare the numbers directly (different δ, whole-number token). |
| `far`: MISMATCH tens/ones vs y ≈ ORIGINAL vs y, and vs y′ ≈ proximity null ≈ chance | The graft fixes those digits; the recipient doesn't use c from its prompt. Answer transfer. |
| `far`: MISMATCH tens/ones vs y′ clearly above both the proximity null and chance | The recipient computes those digits from the c on its screen: computation transfer by the paper's definition. |
| SHUFFLED vs y′ above chance | The recipient computes from its prompt even when the graft carries an unrelated answer. |
| The task map adapts more than the `:donor` bottleneck at similar ORIGINAL | The task map passes something beyond the answer that lets the recipient use its operands. |
| y′-agreement high only where P′ (or its twin) is in the map's training set | Map memorisation (the donor states for P and P′ are nearly identical), not recipient computation. Use the `not_in_this_maps_train` rows. |
| MATCHED vs y′ low (use `not_in_this_maps_train`) | The pipeline can't emit y′ even when the donor saw P′, so low agreement with y′ under MISMATCH is weak evidence. |
| SHUFFLED vs its donor low | The positive control failed for that map; Part D says nothing about it. |

Part D only detects completion that depends on c, because a and b are the same in P and P′.

**What the run can't settle.** If Part C still falls short with no masked-code or Part D effect, the bottleneck is inconclusive (the answer may be in a form the recipient decodes but probes don't), not evidence against the thesis. Before running, expect the reader to be the bottleneck: the paper's per-digit probe reads the tens digit at 0.153 against a 0.121 majority (trained on 321 donor-solved problems). The `:donor` readers here train on almost every reader-half problem (1,500 in Part B, 9,000 in Part C), each labelled with whatever answer the donor gave, right on ~32% of them.

In [18]:
# === CELL F1: follow-up helpers (run after S1, same kernel) ===
for _need in ("train", "evalp", "unsolv", "X9t", "X2t", "X9e", "task_maps", "DONOR_TOP_TRAIN",
              "_train_writer", "_twin", "_answer_of_token", "PER_ITEM", "BN_MODELS"):
    assert _need in globals(), f"missing {_need}: run Parts A-D (cells above) first"
FU = {}
FU_PER_ITEM = {}

def fu_save(tag):
    fn = "llama_followup_results" + ("_SMOKE" if SMOKE_TEST else "") + ".json"
    try:
        with open(fn, "w") as fh:
            json.dump({"_last_completed": tag, "followup": FU, "per_item": FU_PER_ITEM}, fh, indent=1, default=str)
        print(f"[followup checkpoint after {tag} -> {fn}]")
    except Exception as e:
        print("followup checkpoint failed:", e)

_rt = lambda v: round(sum(v) / len(v), 3) if len(v) else None

@torch.inference_mode()
def fu_confer(map_fn, items, X9rows):
    """First-token conferral on ANY item list (training or eval problems): graft map_fn(donor state)
    at the recipient's last prompt position, score argmax == the item's gold first token."""
    assert X9rows.shape[0] == len(items), (X9rows.shape, len(items))
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(items), ARITH_BATCH):
            chunk = items[i:i + ARITH_BATCH]
            v = map_fn(X9rows[i:i + len(chunk)])
            assert v.shape[0] == len(chunk)          # patch_vec_batch silently skips a size mismatch
            _graft["vec"] = v
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == chunk[k]["tok"] for k in range(len(chunk))]
    finally:
        h.remove(); _graft["vec"] = None
    return out

def fu_mk(W, b, mu):
    """Affine map from donor states: (x9 - mu) @ W + b, on DEVICE."""
    return lambda x9, W=W, b=b, mu=mu: (x9.to(DEVICE) - mu) @ W + b

def _trunc(W, r):
    U, S, Vh = torch.linalg.svd(W.float(), full_matrices=False)
    return (U[:, :r] * S[:r]) @ Vh[:r]

def fu_train_map(train_items, X9tr, X2tr, idx, rank=None, factor_full=False, lr=None, bias_lr=None,
                 init="ridge", init_rank=None, seed_base=None, epochs=None, tag=""):
    """The Part B/C matched-map recipe (_train_matched_task: ridge on idx, then CE through the frozen
    recipient, Adam), with one knob changed at a time:
      rank=None              full-rank W. Defaults reproduce Part B/C's matched map (seed_base 2000)
      rank=r                 W = A @ B of rank r from the rank-r SVD of the ridge, then trained.
                             seed_base=3000 reproduces Part B/C's rank-27 map
      factor_full=True       W = A @ B at FULL rank (equals the ridge at init): factorised, no rank limit
      init='trunc'           full-rank W that STARTS at the rank-init_rank truncation of the ridge
      init='zero'            full-rank W that starts at 0 (bias starts at the recipient mean, as always)
      lr / bias_lr           Adam step for W (or A, B) / for the bias. Both default to BN_WRITER_LR
    Returns a dict: fn (map), ce (last-epoch mean training CE), W0/b0 (start), W/b (trained), mu."""
    lr = BN_WRITER_LR if lr is None else lr
    bias_lr = BN_WRITER_LR if bias_lr is None else bias_lr
    epochs = BN_WRITER_EPOCHS if epochs is None else epochs
    factored = factor_full or rank is not None
    assert not (factored and init != "ridge"), "factorised maps start from the ridge"
    if seed_base is None: seed_base = 3000 if rank is not None else 2000
    torch.manual_seed(BN_SEED)
    _mu9, _mu2, _Wr0 = fit_ridge(X9tr[idx], X2tr[idx])
    _mu9d = _mu9.to(DEVICE)
    if init == "trunc": W_init = _trunc(_Wr0, init_rank)
    elif init == "zero": W_init = torch.zeros_like(_Wr0)
    else: W_init = _Wr0
    bm = _mu2.clone().to(DEVICE).float().requires_grad_(True)
    b0 = bm.detach().clone()
    if not factored:
        params = [W_init.clone().to(DEVICE).requires_grad_(True)]
        Wfn = lambda: params[0]
    else:
        U, Sv, Vh = torch.linalg.svd(_Wr0, full_matrices=False)
        r = Sv.numel() if factor_full else min(rank, Sv.numel()); sq = Sv[:r].sqrt()
        params = [(U[:, :r] * sq).to(DEVICE).requires_grad_(True),
                  (sq[:, None] * Vh[:r]).to(DEVICE).requires_grad_(True)]
        Wfn = lambda: params[0] @ params[1]
    with torch.no_grad(): W0 = Wfn().detach().clone()
    # two param groups so a smaller W step does not also shrink the bias step (identical to one group
    # when lr == bias_lr: Adam is per-parameter)
    opt = torch.optim.Adam([{"params": params, "lr": lr}, {"params": [bm], "lr": bias_lr}])
    model_2b.requires_grad_(False)
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    ce_last = float("nan")
    try:
        for ep in range(epochs):
            order = list(idx); random.Random(seed_base + ep).shuffle(order)
            tot, seen = 0.0, 0
            for s in range(0, len(order), ARITH_BATCH):
                sub = order[s:s + ARITH_BATCH]
                _graft["vec"] = (X9tr[sub].to(DEVICE) - _mu9d) @ Wfn() + bm
                ids, m = left_pad([train_items[k]["ids"] for k in sub], tokenizer.pad_token_id)
                logits = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train_items[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(logits, tgt)
                opt.zero_grad(); loss.backward(); opt.step()
                tot += float(loss.item()) * len(sub); seen += len(sub)
            ce_last = tot / max(1, seen)
    finally:
        h.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
    with torch.no_grad(): Wm = Wfn().detach()
    bm = bm.detach()
    print(f"    trained {tag}: last-epoch training CE {ce_last:.3f}")
    return {"fn": fu_mk(Wm, bm, _mu9d), "ce": ce_last, "W0": W0, "b0": b0, "W": Wm, "b": bm, "mu": _mu9d,
            "W_ridge": _Wr0.to(DEVICE), "b_ridge": _mu2.to(DEVICE).float()}

@torch.no_grad()
def fu_change(M):
    """How far training moved the map: relative change of W, change of the bias, and the stable rank
    (||dW||_F^2 / ||dW||_2^2) of the weight update."""
    dW = (M["W"] - M["W0"]).float()
    n0 = float(M["W0"].float().norm())
    s1 = float(torch.linalg.matrix_norm(dW, ord=2)) if float(dW.norm()) > 0 else 0.0
    return {"rel_change_W": round(float(dW.norm()) / n0, 4) if n0 > 0 else None,
            "abs_change_W": round(float(dW.norm()), 3),
            "change_bias": round(float((M["b"] - M["b0"]).norm()), 3),
            "bias_norm": round(float(M["b"].norm()), 3),
            "stable_rank_of_update": round(float(dW.norm()) ** 2 / s1 ** 2, 2) if s1 > 0 else None}

def fu_constant(train_items, X2tr, idx, tag=""):
    """The Part B/C 'uniform scores (bias only)' writer, retrained exactly (_train_writer, digit width 30):
    one vector, identical for every problem."""
    Bu, bu = _train_writer(lambda sub: torch.full((len(sub), 30), 0.1, device=DEVICE), 30,
                           train_items, X2tr, idx, tag)
    v = (torch.full((1, 30), 0.1, device=DEVICE) @ Bu + bu)          # [1, d2]
    return (lambda x9, v=v: v.expand(x9.shape[0], -1)), v

def fu_split(train_items, donor_top, native_top, seed=BN_SEED):
    """Part B/C's reader/writer split (same shuffle when seed = BN_SEED), each half filtered like the bin."""
    ridx = list(range(len(train_items))); random.Random(seed).shuffle(ridx)
    cut = int(len(ridx) * BN_READER_FRAC); R_, W_ = ridx[:cut], ridx[cut:]
    tok = lambda i: train_items[i]["tok"]
    like = lambda i: donor_top[i] == tok(i) and native_top[i] != tok(i)
    w_ex = {train_items[i]["expr"] for i in W_}
    w_tok = {tok(i) for i in W_}
    unseen = [i for i in R_ if like(i)]
    return {"R": R_, "W": W_, "seed": seed,
            "all_binlike": [i for i in range(len(train_items)) if like(i)],
            "seen": [i for i in W_ if like(i)],
            "unseen": unseen,
            # held-out subsets: operand-swap twin (same answer) NOT in the writer half; answer class
            # that DOES occur in the writer half (a map cannot be expected to emit a class it never saw)
            "unseen_notwin": [i for i in unseen if _twin(train_items[i]["expr"]) not in w_ex],
            "unseen_ans_in_writer": [i for i in unseen if tok(i) in w_tok],
            # reader-half problems the DONOR gets wrong (recipient also wrong): the donor-right filter
            # should not matter to a constant, which never sees the donor
            "unseen_donor_wrong": [i for i in R_ if donor_top[i] != tok(i) and native_top[i] != tok(i)]}

def _gap(s, u):
    """seen minus held-out, with a normal-approximation 95% interval (two independent item sets)."""
    if not s or not u: return None
    ps, pu = sum(s) / len(s), sum(u) / len(u)
    se = math.sqrt(ps * (1 - ps) / len(s) + pu * (1 - pu) / len(u))
    return f"{ps - pu:+.3f} [{ps - pu - 1.96 * se:+.3f}, {ps - pu + 1.96 * se:+.3f}]"

def fu_eval(fn, train_items, X9tr, sp, key):
    """Seen / held-out / eval-bin conferral for one map, plus held-out subsets. Saves per-item eval-bin."""
    s = fu_confer(fn, [train_items[i] for i in sp["seen"]], X9tr[sp["seen"]])
    u = fu_confer(fn, [train_items[i] for i in sp["unseen"]], X9tr[sp["unseen"]])
    b = fu_confer(fn, [evalp[j] for j in unsolv], X9e[unsolv])
    pos = {i: k for k, i in enumerate(sp["unseen"])}
    sub = lambda ks: [u[pos[i]] for i in ks]
    out = {"seen_train": fmt(wilson_bools(s)), "held_out_train": fmt(wilson_bools(u)),
           "eval_bin": fmt(wilson_bools(b)), "gap_seen_minus_heldout": _gap(s, u),
           "held_out_no_twin": fmt(wilson_bools(sub(sp["unseen_notwin"]))) if sp["unseen_notwin"] else None,
           "held_out_answer_in_writer_half": fmt(wilson_bools(sub(sp["unseen_ans_in_writer"])))
                                             if sp["unseen_ans_in_writer"] else None,
           "gap_seen_minus_heldout_answer_in_writer_half": _gap(s, sub(sp["unseen_ans_in_writer"])),
           "n_seen": len(s), "n_held_out": len(u)}
    FU_PER_ITEM[f"{key}/eval_bin"] = [int(x) for x in b]
    return out, b

def _line(name, r):
    print(f"  {name:30s} seen {r['seen_train']} | held-out {r['held_out_train']} | eval bin {r['eval_bin']} "
          f"| gap {r['gap_seen_minus_heldout']}")

def _repro(store, label, old, new):
    agree = sum(int(x) == int(y) for x, y in zip(old, new))
    store[label] = {"before": _rt(old), "now": _rt(new), "items_agreeing": f"{agree}/{len(old)}"}
    print(f"  reproduces earlier {label}: {_rt(old)} before vs {_rt(new)} now ({agree}/{len(old)} items agree)")

# the recipient's own top-1 on every 3k training problem (defines the bin-like training subsets)
NATIVE_TOP_TRAIN = states_and_top(model_2b, [], [p["ids"] for p in train])[1]
SP3 = fu_split(train, DONOR_TOP_TRAIN, NATIVE_TOP_TRAIN)
print(f"3k: bin-like training problems {len(SP3['all_binlike'])}/{len(train)} | writer half (seen) "
      f"{len(SP3['seen'])} | reader half (held-out) {len(SP3['unseen'])}: no twin in writer half "
      f"{len(SP3['unseen_notwin'])}, answer occurs in writer half {len(SP3['unseen_ans_in_writer'])}")
FU["config"] = {"writer_epochs": BN_WRITER_EPOCHS, "writer_lr": BN_WRITER_LR, "seed": BN_SEED,
                "split": "Part B/C reader/writer split; seen = writer half, held-out = reader half, both "
                         "filtered like the eval bin (donor top-1 right, recipient top-1 wrong)"}
fu_save("F1")

3k: bin-like training problems 914/3000 | writer half (seen) 470 | reader half (held-out) 444: no twin in writer half 427, answer occurs in writer half 398
[followup checkpoint after F1 -> llama_followup_results.json]


In [19]:
# === CELL F2: 3k — does the full-rank map overfit, and what makes rank 27 different? ===
# Every trained map: the 1,500-problem writer half, 8 epochs, Adam, like Part B's matched map. Each row
# changes ONE thing from 'full_rank'. Training order is the same as full_rank (seed_base 2000) except
# 'rank_27', which keeps Part B's order (3000) so it reproduces Part B; 'rank_27_order2000' is the
# same map in full_rank's order.
t0 = time.time()
F2 = {"maps": {}, "weight_change": {}}
F2_M = {}
_variants = [
    ("full_rank",               dict()),                                # = Part B matched map (0.198)
    ("rank_27",                 dict(rank=27)),                         # = Part B rank-27 map (0.521)
    ("rank_27_order2000",       dict(rank=27, seed_base=2000)),
    ("factorised_full_rank",    dict(factor_full=True)),                # A @ B, no rank limit, ridge start
    ("full_rank_start_trunc27", dict(init="trunc", init_rank=27)),      # full rank, rank-27 starting point
    ("full_rank_start_zero",    dict(init="zero")),                     # full rank, starts at W = 0
    ("full_rank_W_lr1e-4",      dict(lr=1e-4)),                         # 10x smaller W step, bias step unchanged
    ("rank_8",                  dict(rank=8, seed_base=2000)),
    ("rank_64",                 dict(rank=64, seed_base=2000)),
    ("rank_256",                dict(rank=256, seed_base=2000)),
]
for name, kw in _variants:
    M = fu_train_map(train, X9t, X2t, SP3["W"], tag=f"3k/{name}", **kw)
    res, _ = fu_eval(M["fn"], train, X9t, SP3, f"3k/{name}")
    res["train_CE_last_epoch"] = round(M["ce"], 4)
    F2["maps"][name] = res; F2["weight_change"][name] = fu_change(M); F2_M[name] = M
    _line(name, res)

# no training: which part of the trained map does the work, and the paper's truncation method
FR, R27 = F2_M["full_rank"], F2_M["rank_27"]
_untrained = {
    "ridge_untrained":                fu_mk(FR["W_ridge"], FR["b_ridge"], FR["mu"]),   # = recon map on the writer half
    "ridge_trunc27_untrained":        fu_mk(_trunc(FR["W_ridge"], 27), FR["b_ridge"], FR["mu"]),
    "full_rank_trained_W_start_bias": fu_mk(FR["W"], FR["b0"], FR["mu"]),
    "full_rank_start_W_trained_bias": fu_mk(FR["W0"], FR["b"], FR["mu"]),
    "rank_27_trained_W_start_bias":   fu_mk(R27["W"], R27["b0"], R27["mu"]),
    "rank_27_start_W_trained_bias":   fu_mk(R27["W0"], R27["b"], R27["mu"]),
    "full_rank_trained_then_truncated_27": fu_mk(_trunc(FR["W"], 27), FR["b"], FR["mu"]),   # the paper's E10 method
}
for name, fn in _untrained.items():
    res, _ = fu_eval(fn, train, X9t, SP3, f"3k/{name}")
    F2["maps"][name] = res; _line(name, res)

# the constant (no-information) vector and two nulls for it
C3_fn, VCONST_3K = fu_constant(train, X2t, SP3["W"], tag="3k/constant")
_shuf = [dict(p) for p in train]                                         # same prompts, writer-half labels permuted
_perm = list(SP3["W"]); random.Random(BN_SEED + 11).shuffle(_perm)
for i_, j_ in zip(SP3["W"], _perm): _shuf[i_]["tok"] = train[j_]["tok"]
C3s_fn, _ = fu_constant(_shuf, X2t, SP3["W"], tag="3k/constant_shuffled_labels")
_mean_state = X2t.mean(0).to(DEVICE).float()[None]                       # the constant writer's starting bias
for name, fn in (("constant", C3_fn), ("constant_shuffled_labels", C3s_fn),
                 ("mean_recipient_state_untrained", lambda x9: _mean_state.expand(x9.shape[0], -1))):
    res, _ = fu_eval(fn, train, X9t, SP3, f"3k/{name}")
    dw = fu_confer(fn, [train[i] for i in SP3["unseen_donor_wrong"]], X9t[SP3["unseen_donor_wrong"]])
    res["held_out_donor_WRONG"] = fmt(wilson_bools(dw)) if dw else None
    F2["maps"][name] = res; _line(name, res)
    print(f"  {'':30s} held-out problems the donor gets WRONG: {res['held_out_donor_WRONG']}")
F2_M["constant"] = {"fn": C3_fn}

# Part A's task map (seed 0, trained on ALL 3,000): every bin-like training problem is seen
_s = fu_confer(map_task(0), [train[i] for i in SP3["all_binlike"]], X9t[SP3["all_binlike"]])
F2["partA_task_map_seed0_all3000"] = {"seen_train": fmt(wilson_bools(_s)), "n_seen": len(_s),
                                      "eval_bin": RESULTS["arithmetic"]["task_unsolv_first_pooled"]}
print(f"  Part A task map (seed 0, all 3,000): seen {fmt(wilson_bools(_s))} | eval bin "
      f"{RESULTS['arithmetic']['task_unsolv_first_pooled']}")

# reproduction of Part B (GPU kernels are not bit-deterministic, so a few items may differ)
F2["reproduces_partB"] = {}
for name, key in (("full_rank", "matched_task_first"), ("rank_27", f"matched_rank{BN_MATCHED_RANK}_task_first")):
    old = PER_ITEM.get("base", {}).get(key)
    if old is not None: _repro(F2["reproduces_partB"], name, old, FU_PER_ITEM[f"3k/{name}/eval_bin"])
old = PER_ITEM.get("base", {}).get("digits", {}).get("uniform_first")
if old is not None: _repro(F2["reproduces_partB"], "constant", old, FU_PER_ITEM["3k/constant/eval_bin"])
for name in ("full_rank", "rank_27", "rank_27_order2000", "factorised_full_rank", "full_rank_start_trunc27"):
    print(f"  weight change {name:24s} {F2['weight_change'][name]}")
F2["_minutes"] = round((time.time() - t0) / 60, 1)
FU["F2_3k"] = F2
fu_save("F2")

    trained 3k/full_rank: last-epoch training CE 0.252
  full_rank                      seen 0.985 [0.970, 0.993] | held-out 0.164 [0.133, 0.202] | eval bin 0.198 [0.169, 0.231] | gap +0.821 [+0.785, +0.857]
    trained 3k/rank_27: last-epoch training CE 0.337
  rank_27                        seen 0.989 [0.975, 0.995] | held-out 0.473 [0.427, 0.519] | eval bin 0.521 [0.482, 0.560] | gap +0.516 [+0.469, +0.564]
    trained 3k/rank_27_order2000: last-epoch training CE 0.247
  rank_27_order2000              seen 0.991 [0.978, 0.997] | held-out 0.520 [0.474, 0.566] | eval bin 0.554 [0.515, 0.593] | gap +0.471 [+0.424, +0.518]
    trained 3k/factorised_full_rank: last-epoch training CE 1.420
  factorised_full_rank           seen 0.855 [0.821, 0.884] | held-out 0.041 [0.026, 0.063] | eval bin 0.081 [0.063, 0.106] | gap +0.815 [+0.778, +0.851]
    trained 3k/full_rank_start_trunc27: last-epoch training CE 0.189
  full_rank_start_trunc27        seen 0.989 [0.975, 0.995] | held-out 0.146 [0.117

In [20]:
# === CELL F3: 3k — seeds for the full-rank and rank-27 matched maps ===
# (a) training order: initialisation is deterministic (ridge / its SVD), so this varies only the order.
#     Seed 0 is F2's map. Order seeds are base + 100*k, so the two families never share an order.
# (b) reader/writer split: a different 1,500-problem writer half (and held-out half), seeds 1 and 2.
t0 = time.time()
F3 = {"order_seeds": {}, "split_seeds": {}}
for name, kw, base in (("full_rank", dict(), 2000), ("rank_27", dict(rank=27), 3000)):
    vals = [_rt(FU_PER_ITEM[f"3k/{name}/eval_bin"])]
    for sd in range(1, 5):
        M = fu_train_map(train, X9t, X2t, SP3["W"], seed_base=base + 100 * sd, tag=f"3k/{name}/order{sd}", **kw)
        b = fu_confer(M["fn"], [evalp[j] for j in unsolv], X9e[unsolv])
        FU_PER_ITEM[f"3k/{name}/order{sd}/eval_bin"] = [int(x) for x in b]
        vals.append(_rt(b)); del M
    F3["order_seeds"][name] = {"eval_bin_per_seed": vals, "across_seed": fmt(across_seed_ci(vals))}
    print(f"  order seeds {name:10s} {vals} -> {F3['order_seeds'][name]['across_seed']}")
for ss in (1, 2):
    sp = fu_split(train, DONOR_TOP_TRAIN, NATIVE_TOP_TRAIN, seed=BN_SEED + ss)
    for name, kw in (("full_rank", dict()), ("rank_27", dict(rank=27))):
        M = fu_train_map(train, X9t, X2t, sp["W"], tag=f"3k/{name}/split{ss}", **kw)
        res, _ = fu_eval(M["fn"], train, X9t, sp, f"3k/{name}/split{ss}")
        F3["split_seeds"][f"{name}/split{ss}"] = res; _line(f"{name}/split{ss}", res); del M
F3["_minutes"] = round((time.time() - t0) / 60, 1)
FU["F3_3k_seeds"] = F3
fu_save("F3")

    trained 3k/full_rank/order1: last-epoch training CE 0.151
    trained 3k/full_rank/order2: last-epoch training CE 0.161
    trained 3k/full_rank/order3: last-epoch training CE 0.152
    trained 3k/full_rank/order4: last-epoch training CE 0.163
  order seeds full_rank  [0.198, 0.174, 0.165, 0.192, 0.203] -> 0.186 [0.166, 0.207]
    trained 3k/rank_27/order1: last-epoch training CE 0.353
    trained 3k/rank_27/order2: last-epoch training CE 0.313
    trained 3k/rank_27/order3: last-epoch training CE 0.223
    trained 3k/rank_27/order4: last-epoch training CE 0.251
  order seeds rank_27    [0.521, 0.519, 0.498, 0.518, 0.556] -> 0.522 [0.496, 0.548]
    trained 3k/full_rank/split1: last-epoch training CE 0.190
  full_rank/split1               seen 0.991 [0.978, 0.997] | held-out 0.193 [0.159, 0.232] | eval bin 0.201 [0.172, 0.234] | gap +0.798 [+0.761, +0.836]
    trained 3k/rank_27/split1: last-epoch training CE 0.249
  rank_27/split1                 seen 0.996 [0.984, 0.999] | held-o

In [21]:
# === CELL F4: 18k — the same checks with a 9,000-problem writer half ===
RUN_FU_18K = True          # False skips this cell (~20 min)
F4_M = {}
if RUN_FU_18K and globals().get("TASKBIG") is not None:
    t0 = time.time()
    F4 = {"maps": {}, "weight_change": {}}
    nested = [p["expr"] for p in train_big[:len(train)]] == [p["expr"] for p in train]
    if nested:
        NATIVE_TOP_BIG = list(NATIVE_TOP_TRAIN) + list(
            states_and_top(model_2b, [], [p["ids"] for p in train_big[len(train):]])[1])
    else:
        NATIVE_TOP_BIG = states_and_top(model_2b, [], [p["ids"] for p in train_big])[1]
    SP18 = fu_split(train_big, DONOR_TOP_BIG, NATIVE_TOP_BIG)
    print(f"18k: bin-like {len(SP18['all_binlike'])}/{len(train_big)} | seen {len(SP18['seen'])} | held-out "
          f"{len(SP18['unseen'])} (no twin {len(SP18['unseen_notwin'])}, answer in writer half "
          f"{len(SP18['unseen_ans_in_writer'])})")
    for name, kw in (("full_rank", dict()), ("rank_27", dict(rank=27)),
                     ("factorised_full_rank", dict(factor_full=True)),
                     ("full_rank_start_trunc27", dict(init="trunc", init_rank=27))):
        M = fu_train_map(train_big, X9b, X2b, SP18["W"], tag=f"18k/{name}", **kw)
        res, _ = fu_eval(M["fn"], train_big, X9b, SP18, f"18k/{name}")
        res["train_CE_last_epoch"] = round(M["ce"], 4)
        F4["maps"][name] = res; F4["weight_change"][name] = fu_change(M); F4_M[name] = M
        _line(name, res)
    FR18, R2718 = F4_M["full_rank"], F4_M["rank_27"]
    for name, fn in (("full_rank_trained_W_start_bias", fu_mk(FR18["W"], FR18["b0"], FR18["mu"])),
                     ("full_rank_start_W_trained_bias", fu_mk(FR18["W0"], FR18["b"], FR18["mu"])),
                     ("rank_27_trained_W_start_bias",   fu_mk(R2718["W"], R2718["b0"], R2718["mu"])),
                     ("rank_27_start_W_trained_bias",   fu_mk(R2718["W0"], R2718["b"], R2718["mu"])),
                     ("full_rank_trained_then_truncated_27", fu_mk(_trunc(FR18["W"], 27), FR18["b"], FR18["mu"]))):
        res, _ = fu_eval(fn, train_big, X9b, SP18, f"18k/{name}")
        F4["maps"][name] = res; _line(name, res)
    C18_fn, VCONST_18K = fu_constant(train_big, X2b, SP18["W"], tag="18k/constant")
    res, _ = fu_eval(C18_fn, train_big, X9b, SP18, "18k/constant")
    dw = fu_confer(C18_fn, [train_big[i] for i in SP18["unseen_donor_wrong"]], X9b[SP18["unseen_donor_wrong"]])
    res["held_out_donor_WRONG"] = fmt(wilson_bools(dw)) if dw else None
    F4["maps"]["constant"] = res; F4_M["constant"] = {"fn": C18_fn}; _line("constant", res)
    print(f"  {'':30s} held-out problems the donor gets WRONG: {res['held_out_donor_WRONG']}")
    _s = fu_confer(TASKBIG, [train_big[i] for i in SP18["all_binlike"]], X9b[SP18["all_binlike"]])
    F4["partC_task_map_all18000"] = {"seen_train": fmt(wilson_bools(_s)), "n_seen": len(_s),
                                     "eval_bin": RESULTS["scaled_reference_maps"]["task_first_18k"]}
    print(f"  Part C task map (all 18,000): seen {fmt(wilson_bools(_s))} | eval bin "
          f"{RESULTS['scaled_reference_maps']['task_first_18k']}")
    F4["reproduces_partC"] = {}
    for name, key in (("full_rank", "matched_task_first"), ("rank_27", f"matched_rank{BN_MATCHED_RANK}_task_first")):
        old = PER_ITEM.get("scaled", {}).get(key)
        if old is not None: _repro(F4["reproduces_partC"], name, old, FU_PER_ITEM[f"18k/{name}/eval_bin"])
    old = PER_ITEM.get("scaled", {}).get("digits", {}).get("uniform_first")
    if old is not None: _repro(F4["reproduces_partC"], "constant", old, FU_PER_ITEM["18k/constant/eval_bin"])
    for name in F4["weight_change"]: print(f"  weight change {name:24s} {F4['weight_change'][name]}")
    F4["_minutes"] = round((time.time() - t0) / 60, 1)
    FU["F4_18k"] = F4
    fu_save("F4")
else:
    print("F4 skipped")

18k: bin-like 5458/18000 | seen 2709 | held-out 2749 (no twin 1819, answer in writer half 2733)
    trained 18k/full_rank: last-epoch training CE 0.528
  full_rank                      seen 0.909 [0.898, 0.919] | held-out 0.575 [0.557, 0.593] | eval bin 0.559 [0.520, 0.598] | gap +0.334 [+0.313, +0.355]
    trained 18k/rank_27: last-epoch training CE 0.277
  rank_27                        seen 0.960 [0.952, 0.967] | held-out 0.869 [0.856, 0.881] | eval bin 0.871 [0.842, 0.895] | gap +0.091 [+0.077, +0.106]
    trained 18k/factorised_full_rank: last-epoch training CE 2.050
  factorised_full_rank           seen 0.593 [0.574, 0.611] | held-out 0.239 [0.223, 0.255] | eval bin 0.220 [0.190, 0.255] | gap +0.354 [+0.330, +0.379]
    trained 18k/full_rank_start_trunc27: last-epoch training CE 0.562
  full_rank_start_trunc27        seen 0.906 [0.894, 0.916] | held-out 0.555 [0.537, 0.574] | eval bin 0.530 [0.491, 0.569] | gap +0.350 [+0.328, +0.372]
  full_rank_trained_W_start_bias seen 0.680 [

In [22]:
# === CELL F5: Part D (prompt swap) on the new maps ===
# run_perturb is Part D's own function, so every condition and null is computed exactly as before.
# The constant vector is a POSITIVE CONTROL here, not a leakage test: its graft is the same whatever the
# donor saw, so MISMATCH-vs-y is ~0 by construction, and MISMATCH-vs-y' measures how often the recipient
# computes P''s answer by itself with that vector in place.
assert "run_perturb" in globals() and "DELTAS" in globals(), "Part D must have run in this kernel"
t0 = time.time()
NEW_MAPS = {"rank27_matched_3k": F2_M["rank_27"]["fn"], "fullrank_matched_3k": F2_M["full_rank"]["fn"],
            "fullrank_start_trunc27_3k": F2_M["full_rank_start_trunc27"]["fn"],
            "constant_3k": F2_M["constant"]["fn"]}
if F4_M:
    NEW_MAPS.update({"rank27_matched_18k": F4_M["rank_27"]["fn"], "fullrank_matched_18k": F4_M["full_rank"]["fn"],
                     "constant_18k": F4_M["constant"]["fn"]})
F5 = {"rules": {rule: run_perturb(rule, NEW_MAPS) for rule in P_RULES}}
for rule in P_RULES:
    F5["rules"][rule].pop("per_item_task_3k_seed0", None)

# paired ORIGINAL (recipient reads P) vs MISMATCH (reads P'), whole answer, same items: how much of each
# map's conferral needs the recipient to be reading the matching problem
REF_MAPS = {"task_3k_seed0": MAPS["task_3k_seed0"]}
if "task_18k" in MAPS: REF_MAPS["task_18k"] = MAPS["task_18k"]
F5["paired_original_vs_mismatch"] = {}
for rule in P_RULES:
    items, _ = _build_items(rule, P_SHUFFLE_SEED)
    if not items: continue
    src = [p["idx"] for p in items]
    nat = _tops(items, None)
    KS = [k for k in range(len(items)) if nat[k] != items[k]["tok"]]
    orig_items = [evalp[i] for i in src]
    row = {}
    for name, f in {**REF_MAPS, **NEW_MAPS}.items():
        V = f(X9e[src])
        t_mis, t_o = _tops(items, V), _tops(orig_items, V)
        o  = [t_o[k] == items[k]["stale_tok"] for k in KS]        # ORIGINAL: output = y
        mi = [t_mis[k] == items[k]["stale_tok"] for k in KS]      # MISMATCH: output still = y
        a2 = [t_mis[k] == items[k]["tok"] for k in KS]            # MISMATCH: output = y'
        n01, n10, p = mcnemar(o, mi)
        row[name] = {"ORIGINAL_y": fmt(wilson_bools(o)), "MISMATCH_y": fmt(wilson_bools(mi)),
                     "MISMATCH_y2": fmt(wilson_bools(a2)),
                     "share_of_conferral_kept": round(_rt(mi) / _rt(o), 3) if _rt(o) else None,
                     "lost_under_mismatch": n10, "gained_under_mismatch": n01, "mcnemar_p": p}
        print(f"  [{rule}] {name:26s} ORIGINAL y {_rt(o)} | MISMATCH y {_rt(mi)}, y' {_rt(a2)} | "
              f"lost {n10} / gained {n01}, p={p:.2g}")
    F5["paired_original_vs_mismatch"][rule] = row
F5["_minutes"] = round((time.time() - t0) / 60, 1)
FU["F5_partD"] = F5
fu_save("F5")

[D/leading] kept 179/626 bin problems (447 have no valid delta)
[D/leading] P' also natively unsolvable: 159/179 | donor solves P' on 53
  [D/leading] rank27_matched_3k          MISMATCH lead y 0.742 [0.669, 0.804] / y' 0.258 [0.196, 0.331] | whole y 0.126 [0.083, 0.186] / y' 0.119 [0.078, 0.179]
  [D/leading] fullrank_matched_3k        MISMATCH lead y 0.597 [0.520, 0.671] / y' 0.384 [0.312, 0.461] | whole y 0.063 [0.035, 0.112] / y' 0.044 [0.021, 0.088]
  [D/leading] fullrank_start_trunc27_3k  MISMATCH lead y 0.660 [0.584, 0.729] / y' 0.333 [0.265, 0.410] | whole y 0.069 [0.039, 0.120] / y' 0.050 [0.026, 0.096]
  [D/leading] constant_3k                MISMATCH lead y 0.327 [0.259, 0.403] / y' 0.616 [0.539, 0.688] | whole y 0.006 [0.001, 0.035] / y' 0.226 [0.168, 0.297]
  [D/leading] rank27_matched_18k         MISMATCH lead y 0.849 [0.785, 0.896] / y' 0.151 [0.104, 0.215] | whole y 0.201 [0.146, 0.270] / y' 0.101 [0.063, 0.157]
  [D/leading] fullrank_matched_18k       MISMATCH lead y 0

In [23]:
# === CELL F6: the constant vector — what is it fixing? ===
# Where does the right answer sit in the recipient's OWN (ungrafted) next-token ranking on the bin, and are
# the items the constant gets right ones that were already near misses? Is the native top-1 even a 3-digit
# number, and how far off is it?
@torch.inference_mode()
def fu_logits(items, V=None):
    h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch) if V is not None else None
    out = []
    try:
        for i in range(0, len(items), ARITH_BATCH):
            chunk = items[i:i + ARITH_BATCH]
            _graft["vec"] = V[i:i + len(chunk)].to(DEVICE) if V is not None else None
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            out.append(model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float().cpu())
    finally:
        if h is not None: h.remove()
        _graft["vec"] = None
    return torch.cat(out)

bin_items = [evalp[j] for j in unsolv]
gold = torch.tensor([p["tok"] for p in bin_items])
def _rank_stats(L):
    g = L.gather(1, gold[:, None])
    rank = (L > g).sum(1) + 1                                    # 1 = top (ties counted below)
    ties = int(((L == g).sum(1) > 1).sum())                      # items where another token ties the gold logit
    top = L.argmax(1).tolist()
    ans = [_answer_of_token(t) for t in top]
    err = [abs(a - p["ans"]) for a, p in zip(ans, bin_items) if a is not None]
    return rank, top, ties, [a is not None for a in ans], err
def _rank_summary(rank, ks=None):
    r = rank if ks is None else rank[ks]
    if len(r) == 0: return None
    r = r.float()
    return {"n": int(len(r)), "median_rank_of_right_answer": float(r.median()),
            "share_rank_2": round(float((r == 2).float().mean()), 3),
            "share_rank_le_5": round(float((r <= 5).float().mean()), 3),
            "share_rank_le_20": round(float((r <= 20).float().mean()), 3)}

L_nat = fu_logits(bin_items)
rank_nat, top_nat, ties_nat, is3_nat, err_nat = _rank_stats(L_nat)
c_right = torch.tensor(FU_PER_ITEM["3k/constant/eval_bin"], dtype=torch.bool)
L_c = fu_logits(bin_items, VCONST_3K.expand(len(bin_items), -1))
rank_c, top_c, ties_c, is3_c, err_c = _rank_stats(L_c)
del L_nat, L_c
F6 = {
    "native_accuracy_on_bin_should_be_0": _rt([t == g for t, g in zip(top_nat, gold.tolist())]),
    "native_rank_of_right_answer_all_bin": _rank_summary(rank_nat),
    "native_rank_on_items_the_constant_gets_right": _rank_summary(rank_nat, c_right.nonzero().flatten()),
    "native_rank_on_items_the_constant_gets_wrong": _rank_summary(rank_nat, (~c_right).nonzero().flatten()),
    "native_items_with_a_tie_at_the_gold_logit": ties_nat,
    "native_top1_is_3digit_number": round(float(np.mean(is3_nat)), 3),
    "native_top1_abs_error_median": float(np.median(err_nat)) if err_nat else None,
    "constant_top1_is_3digit_number": round(float(np.mean(is3_c)), 3),
    "constant_top1_abs_error_median": float(np.median(err_c)) if err_c else None,
    "constant_top1_equals_native_top1": _rt([a == b_ for a, b_ in zip(top_c, top_nat)]),
    "constant_rank_of_right_answer_all_bin": _rank_summary(rank_c),
    "constant_items_with_a_tie_at_the_gold_logit": ties_c,
    "constant_conferral_recomputed": _rt([t == g for t, g in zip(top_c, gold.tolist())]),
    "constant_conferral_in_F2": _rt(FU_PER_ITEM["3k/constant/eval_bin"]),
}
print(json.dumps(F6, indent=1))
FU["F6_constant_vector"] = F6
fu_save("F6")

{
 "native_accuracy_on_bin_should_be_0": 0.0,
 "native_rank_of_right_answer_all_bin": {
  "n": 626,
  "median_rank_of_right_answer": 32.0,
  "share_rank_2": 0.032,
  "share_rank_le_5": 0.142,
  "share_rank_le_20": 0.399
 },
 "native_rank_on_items_the_constant_gets_right": {
  "n": 110,
  "median_rank_of_right_answer": 25.0,
  "share_rank_2": 0.045,
  "share_rank_le_5": 0.218,
  "share_rank_le_20": 0.482
 },
 "native_rank_on_items_the_constant_gets_wrong": {
  "n": 516,
  "median_rank_of_right_answer": 33.0,
  "share_rank_2": 0.029,
  "share_rank_le_5": 0.126,
  "share_rank_le_20": 0.382
 },
 "native_items_with_a_tie_at_the_gold_logit": 567,
 "native_top1_is_3digit_number": 0.473,
 "native_top1_abs_error_median": 36.0,
 "constant_top1_is_3digit_number": 1.0,
 "constant_top1_abs_error_median": 20.0,
 "constant_top1_equals_native_top1": 0.019,
 "constant_rank_of_right_answer_all_bin": {
  "n": 626,
  "median_rank_of_right_answer": 5.0,
  "share_rank_2": 0.102,
  "share_rank_le_5": 0.521,


In [24]:
# === CELL F7: summary ===
def _p(x): return x if x is not None else "-"
print("SEEN vs HELD-OUT: first-token conferral on problems the map trained on vs never saw")
for arm, key in (("3k", "F2_3k"), ("18k", "F4_18k")):
    if key not in FU: continue
    print(f"\n  {arm}: writer half trains every map; held-out = reader half; eval bin = Part A's bin")
    print(f"  {'map':36s} {'seen':>21s} {'held-out':>21s} {'eval bin':>21s}  {'gap (seen - held-out)':>24s}  train CE")
    for name, r in FU[key]["maps"].items():
        print(f"  {name:36s} {r['seen_train']:>21s} {r['held_out_train']:>21s} {r['eval_bin']:>21s}  "
              f"{_p(r['gap_seen_minus_heldout']):>24s}  {_p(r.get('train_CE_last_epoch'))}")
    ref = FU[key].get("partA_task_map_seed0_all3000") or FU[key].get("partC_task_map_all18000")
    if ref: print(f"  {'paper task map (all data)':36s} seen {ref['seen_train']} | eval bin {ref['eval_bin']}")
    print("  weight change:", {k: v for k, v in FU[key]["weight_change"].items()})
if "F3_3k_seeds" in FU:
    print("\n3k ORDER SEEDS:", {k: v["across_seed"] for k, v in FU["F3_3k_seeds"]["order_seeds"].items()})
    print("3k SPLIT SEEDS (eval bin):", {k: v["eval_bin"] for k, v in FU["F3_3k_seeds"]["split_seeds"].items()})
if "F5_partD" in FU:
    print("\nPROMPT SWAP, paired (whole answer; ORIGINAL = recipient reads P, MISMATCH = reads P')")
    for rule, row in FU["F5_partD"]["paired_original_vs_mismatch"].items():
        print(f"  rule {rule}")
        for name, r in row.items():
            print(f"    {name:26s} ORIGINAL y {r['ORIGINAL_y']} | MISMATCH y {r['MISMATCH_y']} | y' {r['MISMATCH_y2']} "
                  f"| kept {r['share_of_conferral_kept']} (lost {r['lost_under_mismatch']}, gained "
                  f"{r['gained_under_mismatch']}, p={r['mcnemar_p']:.2g})")
if "F6_constant_vector" in FU:
    f6 = FU["F6_constant_vector"]
    for k in ("native_accuracy_on_bin_should_be_0", "native_rank_of_right_answer_all_bin",
              "native_rank_on_items_the_constant_gets_right", "native_top1_is_3digit_number",
              "native_top1_abs_error_median", "constant_top1_equals_native_top1",
              "constant_conferral_recomputed", "constant_conferral_in_F2"):
        print(f"  {k}: {f6[k]}")
fu_save("F7")

SEEN vs HELD-OUT: first-token conferral on problems the map trained on vs never saw

  3k: writer half trains every map; held-out = reader half; eval bin = Part A's bin
  map                                                   seen              held-out              eval bin     gap (seen - held-out)  train CE
  full_rank                             0.985 [0.970, 0.993]  0.164 [0.133, 0.202]  0.198 [0.169, 0.231]   +0.821 [+0.785, +0.857]  0.2522
  rank_27                               0.989 [0.975, 0.995]  0.473 [0.427, 0.519]  0.521 [0.482, 0.560]   +0.516 [+0.469, +0.564]  0.3371
  rank_27_order2000                     0.991 [0.978, 0.997]  0.520 [0.474, 0.566]  0.554 [0.515, 0.593]   +0.471 [+0.424, +0.518]  0.2475
  factorised_full_rank                  0.855 [0.821, 0.884]  0.041 [0.026, 0.063]  0.081 [0.063, 0.106]   +0.815 [+0.778, +0.851]  1.4204
  full_rank_start_trunc27               0.989 [0.975, 0.995]  0.146 [0.117, 0.182]  0.169 [0.142, 0.201]   +0.843 [+0.809, +0.877]  0.

In [25]:
# === CELL F8: prompt swap on the best map (full rank, 10x smaller weight step) and the rank ladder ===
# Run after F7, same kernel. Question: when a map generalises better, does it still deliver all three
# digits from the donor, or does it hand a digit to the recipient (as rank 27 hands over the ones digit)?
# 3k maps are reused from F2 (no retraining). The 18k small-step map is trained here (~3-4 min) with F2's
# exact recipe (W lr 1e-4, bias lr 1e-3, 8 epochs, order 2000, ridge start) on F4's 9,000-problem writer
# half. Nothing earlier is modified; results go to FU["F8_small_step_and_rank_ladder"].
#
# Baselines for "the recipient supplies a digit" (MISMATCH output's digit = y''s digit):
#   NATIVE      no graft, recipient reads P'        what the recipient does entirely on its own
#   SHUFFLED    graft from an unrelated problem      the recipient's own digit when the graft is uninformative
#   PROXIMITY   recipient reads P, graft from P      how often y''s digit appears with P' nowhere on screen
#   CHANCE      ~0.10 per digit
# A map hands a digit to the recipient when MISMATCH-vs-y' is well above PROXIMITY and CHANCE AND the same
# digit no longer follows y. Compare with NATIVE/SHUFFLED for how much of the recipient's own ability it uses.
assert "run_perturb" in globals() and "DELTAS" in globals(), "Part D must have run in this kernel"
assert "FU" in globals() and "F2_M" in globals(), "run F1-F7 first (same kernel)"
t0 = time.time()
F8 = {}
_lr_key = "full_rank_W_lr1e-4"

# --- the maps -------------------------------------------------------------------------------------
F8_MAPS = {}
if _lr_key in F2_M:
    _SS3 = F2_M[_lr_key]
else:                                                           # F2 maps lost: retrain identically
    _SS3 = fu_train_map(train, X9t, X2t, SP3["W"], lr=1e-4, tag="3k/" + _lr_key)
F8_MAPS["small_step_3k"] = _SS3["fn"]
F8_MAPS["small_step_Wonly_3k"] = fu_mk(_SS3["W"], _SS3["b0"], _SS3["mu"])     # trained W, starting bias
F8_MAPS["small_step_biasonly_3k"] = fu_mk(_SS3["W0"], _SS3["b"], _SS3["mu"])  # starting W, trained bias
for nm, key in (("full_rank_3k", "full_rank"), ("rank_27_3k", "rank_27"), ("rank_27_order2000_3k", "rank_27_order2000"),
                ("rank_8_3k", "rank_8"), ("rank_64_3k", "rank_64"), ("rank_256_3k", "rank_256"),
                ("constant_3k", "constant")):
    if key in F2_M: F8_MAPS[nm] = F2_M[key]["fn"]

# sanity: the reused 3k small-step map still gives F2's eval-bin result, item for item
_b = fu_confer(F8_MAPS["small_step_3k"], [evalp[j] for j in unsolv], X9e[unsolv])
_old = FU_PER_ITEM.get(f"3k/{_lr_key}/eval_bin")
if _old is not None:
    _ag = sum(int(x) == int(y) for x, y in zip(_old, _b))
    F8["small_step_3k_matches_F2"] = f"{_ag}/{len(_old)} items agree ({_rt(_old)} in F2, {_rt(_b)} now)"
    print("small-step 3k map vs F2:", F8["small_step_3k_matches_F2"])

if "SP18" in globals() and globals().get("TASKBIG") is not None:
    M18 = fu_train_map(train_big, X9b, X2b, SP18["W"], lr=1e-4, tag="18k/" + _lr_key)
    res, _ = fu_eval(M18["fn"], train_big, X9b, SP18, f"18k/{_lr_key}")
    res["train_CE_last_epoch"] = round(M18["ce"], 4)
    F8["small_step_18k_seen_vs_heldout"] = res
    F8["small_step_18k_weight_change"] = fu_change(M18)
    _line("18k/" + _lr_key, res)
    F8_MAPS["small_step_18k"] = M18["fn"]
    F8_MAPS["small_step_Wonly_18k"] = fu_mk(M18["W"], M18["b0"], M18["mu"])
    F8_MAPS["small_step_biasonly_18k"] = fu_mk(M18["W0"], M18["b"], M18["mu"])
    if "F4_M" in globals():
        for nm, key in (("full_rank_18k", "full_rank"), ("rank_27_18k", "rank_27"), ("constant_18k", "constant")):
            if key in F4_M: F8_MAPS[nm] = F4_M[key]["fn"]
else:
    print("18k skipped (F4 did not run in this kernel)")
print("maps:", list(F8_MAPS))

# --- prompt swap (Part D's own function) ------------------------------------------------------------
F8["rules"] = {rule: run_perturb(rule, F8_MAPS) for rule in P_RULES}
for rule in P_RULES:
    F8["rules"][rule].pop("per_item_task_3k_seed0", None)

# repro: the same F2/F4 maps through run_perturb must give F5's strings exactly (same items, same scoring)
F8["repro_vs_F5"] = {}
for nm_new, nm_old in (("full_rank_3k", "fullrank_matched_3k"), ("rank_27_3k", "rank27_matched_3k"),
                       ("full_rank_18k", "fullrank_matched_18k"), ("rank_27_18k", "rank27_matched_18k")):
    for rule in P_RULES:
        a = F8["rules"][rule]["maps"].get(nm_new, {}).get("MISMATCH")
        b_ = FU.get("F5_partD", {}).get("rules", {}).get(rule, {}).get("maps", {}).get(nm_old, {}).get("MISMATCH")
        if a is not None and b_ is not None:
            F8["repro_vs_F5"][f"{nm_new}/{rule}"] = (a == b_)
print("same results as F5 for the shared maps:", F8["repro_vs_F5"])

# --- paired per-item tests on the 'far' items: whole answer and each digit, vs y and vs y' -------------
def _digit_of_tok(t, n):
    s = tokenizer.decode([t]).strip()
    return int(s[n]) if (len(s) == 3 and s.isdigit()) else None

F8["paired"] = {}
F8_PER = {}
for rule in P_RULES:
    items, _ = _build_items(rule, P_SHUFFLE_SEED)
    if not items: continue
    src = [p["idx"] for p in items]
    nat = _tops(items, None)
    KS = [k for k in range(len(items)) if nat[k] != items[k]["tok"]]
    orig_items = [evalp[i] for i in src]
    per, row = {}, {}
    for name, f in F8_MAPS.items():
        V = f(X9e[src])
        t_mis, t_o = _tops(items, V), _tops(orig_items, V)
        d = {"orig_whole_y": [t_o[k] == items[k]["stale_tok"] for k in KS],
             "mis_whole_y":  [t_mis[k] == items[k]["stale_tok"] for k in KS],
             "mis_whole_y2": [t_mis[k] == items[k]["tok"] for k in KS]}
        for n, dn in ((0, "hundreds"), (1, "tens"), (2, "ones")):
            ks_n = [k for k in KS if _digits(items[k]["y"])[n] != _digits(items[k]["y2"])[n]]
            d[f"mis_{dn}_y"]  = [_digit_of_tok(t_mis[k], n) == _digits(items[k]["y"])[n] for k in ks_n]
            d[f"mis_{dn}_y2"] = [_digit_of_tok(t_mis[k], n) == _digits(items[k]["y2"])[n] for k in ks_n]
        per[name] = d
        n01, n10, p = mcnemar(d["orig_whole_y"], d["mis_whole_y"])
        row[name] = {"ORIGINAL_y": fmt(wilson_bools(d["orig_whole_y"])), "MISMATCH_y": fmt(wilson_bools(d["mis_whole_y"])),
                     "MISMATCH_y2": fmt(wilson_bools(d["mis_whole_y2"])),
                     "share_of_conferral_kept": (round(_rt(d["mis_whole_y"]) / _rt(d["orig_whole_y"]), 3)
                                                 if _rt(d["orig_whole_y"]) else None),
                     "lost_under_mismatch": n10, "gained_under_mismatch": n01, "mcnemar_p": p}
    # same items, map vs map: does the small-step map hand over the ones digit less than rank 27 does?
    cmp = {}
    for a_, b_ in (("small_step_3k", "rank_27_3k"), ("small_step_3k", "full_rank_3k"),
                   ("small_step_18k", "rank_27_18k"), ("small_step_18k", "full_rank_18k")):
        if a_ in per and b_ in per:
            for key in ("mis_ones_y2", "mis_ones_y", "mis_tens_y2", "mis_whole_y"):
                n01, n10, p = mcnemar(per[a_][key], per[b_][key])
                cmp[f"{a_} vs {b_} / {key}"] = {f"{a_}": _rt(per[a_][key]), f"{b_}": _rt(per[b_][key]),
                                                f"{a_}_only": n10, f"{b_}_only": n01, "mcnemar_p": p}
    F8["paired"][rule] = {"original_vs_mismatch": row, "map_vs_map": cmp}
    F8_PER[rule] = per
FU_PER_ITEM["F8_partD_per_item"] = {r: {m: {k: [int(x) for x in v] for k, v in d.items()} for m, d in per.items()}
                                    for r, per in F8_PER.items()}

# --- one table per rule ----------------------------------------------------------------------------------
def _pt0(s): return s.split(" ")[0] if isinstance(s, str) else "-"
def _digit_table(rule):
    d = F8["rules"][rule]; per = F8_PER.get(rule, {})
    nat = d["NATIVE"]["matches_y2"]
    print(f"\nPROMPT SWAP, rule '{rule}' (recipient reads P', graft from P). Each cell: share of outputs whose "
          f"digit equals the DONOR's problem (y) / the RECIPIENT's prompt (y').")
    print(f"  NATIVE (no graft) vs y': hundreds {_pt0(nat['hundreds'])}  tens {_pt0(nat['tens'])}  ones {_pt0(nat['ones'])}")
    print(f"  {'map':24s} {'ORIG':>5s} | {'whole y/yp':>11s} | {'hund y/yp':>11s} | {'tens y/yp':>11s} | {'ones y/yp':>11s} || "
          f"{'ones yp: prox / shuf / chance':>29s} | {'ones yp, P` not in train':>24s} | kept")
    for name in F8_MAPS:
        r = d["maps"].get(name)
        if r is None: continue
        my, my2 = r["MISMATCH"]["matches_y"], r["MISMATCH"]["matches_y2"]
        prox = r["ORIGINAL"]["matches_y2_PROXIMITY_NULL"]; sh = r["SHUFFLED"]["matches_y2"]
        ch = r.get("CHANCE_null_MISMATCH_outputs", {})
        oot = r["MISMATCH_Pprime_not_in_this_maps_train"]
        kept = F8["paired"].get(rule, {}).get("original_vs_mismatch", {}).get(name, {}).get("share_of_conferral_kept")
        cell = lambda k: f"{_pt0(my[k])}/{_pt0(my2[k])}"
        print(f"  {name:24s} {_pt0(r['ORIGINAL']['matches_y_conferral']['whole']):>5s} | {cell('whole'):>11s} | "
              f"{cell('hundreds'):>11s} | {cell('tens'):>11s} | {cell('ones'):>11s} || "
              f"{_pt0(prox['ones']):>9s} / {_pt0(sh['ones']):>5s} / {_pt0(ch.get('ones')):>5s} | "
              f"{_pt0(oot['matches_y2']['ones']):>16s} (n={oot['n']}) | {kept}")
for rule in P_RULES: _digit_table(rule)
print("\nSAME ITEMS, MAP vs MAP ('far' rule):")
for k, v in F8["paired"].get("far", {}).get("map_vs_map", {}).items():
    print(f"  {k:48s} {v}")

F8["_minutes"] = round((time.time() - t0) / 60, 1)
FU["F8_small_step_and_rank_ladder"] = F8
fu_save("F8")

small-step 3k map vs F2: 626/626 items agree (0.604 in F2, 0.604 now)
    trained 18k/full_rank_W_lr1e-4: last-epoch training CE 0.101
  18k/full_rank_W_lr1e-4         seen 0.989 [0.985, 0.993] | held-out 0.916 [0.905, 0.925] | eval bin 0.919 [0.894, 0.937] | gap +0.074 [+0.063, +0.085]
maps: ['small_step_3k', 'small_step_Wonly_3k', 'small_step_biasonly_3k', 'full_rank_3k', 'rank_27_3k', 'rank_27_order2000_3k', 'rank_8_3k', 'rank_64_3k', 'rank_256_3k', 'constant_3k', 'small_step_18k', 'small_step_Wonly_18k', 'small_step_biasonly_18k', 'full_rank_18k', 'rank_27_18k', 'constant_18k']
[D/leading] kept 179/626 bin problems (447 have no valid delta)
[D/leading] P' also natively unsolvable: 159/179 | donor solves P' on 53
  [D/leading] small_step_3k              MISMATCH lead y 0.824 [0.757, 0.875] / y' 0.176 [0.125, 0.243] | whole y 0.170 [0.119, 0.236] / y' 0.069 [0.039, 0.120]
  [D/leading] small_step_Wonly_3k        MISMATCH lead y 0.818 [0.750, 0.870] / y' 0.176 [0.125, 0.243] | whole y